# Polymer Informatics Atlas

## Notebook 05 — Shared Learning and Predictive Reliability

### Scientific objective

Previous notebooks established that polymer-property prediction depends on both the target property and the region of chemical space being evaluated.

Notebook 03 developed property-specific predictive baselines.

Notebook 04 showed that chemical novelty can substantially reduce predictive reliability for some targets, but chemical similarity alone is not a universal uncertainty measure.

Notebook 05 addresses two related questions.

### Part A — Shared learning

> **Can information from related polymer properties improve prediction through shared representation learning?**

The five target properties do not have equal patterns of data overlap.

In particular, Tc, Density, and Rg form a strongly overlapping measurement block, whereas Tg and FFV are more weakly connected to the other targets.

Shared learning is therefore evaluated only where the data structure provides a scientific justification.

The analysis compares:

- matched single-task neural models;
- a shared-trunk multitask neural model;
- and, where justified, masked multitask learning that can use polymers with partially missing target labels.

The purpose is to identify positive transfer, negligible transfer, or negative transfer rather than assuming that multitask learning must improve prediction.

### Part B — Predictive reliability

> **Can prediction-level uncertainty provide a more direct measure of reliability than chemical similarity alone?**

Model reliability will be evaluated using calibrated prediction intervals.

The aim is not only to predict a property value but also to quantify how strongly the available data supports that prediction.

Reliability analysis will later be related back to the chemical-support results from Notebook 04.

### Methodological principles

All model comparisons are performed using fixed random seeds and explicitly separated training, validation, and evaluation data.

Feature transformations are fitted only on training data.

Shared-learning models are compared using matched input representations and comparable neural architectures so that differences can be attributed to learning strategy rather than representation changes.

Notebook 03 remains the reference for the strongest property-specific classical models.

In [1]:
# ============================================================
# Upload frozen project artifacts
# ============================================================

from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded:
    print(f"  {filename}")

Saving polymer_atlas_notebook04_outputs.zip to polymer_atlas_notebook04_outputs.zip
Saving polymer_atlas_notebook03_outputs.zip to polymer_atlas_notebook03_outputs.zip
Saving polymer_atlas_notebook02_outputs.zip to polymer_atlas_notebook02_outputs.zip

Uploaded files:
  polymer_atlas_notebook04_outputs.zip
  polymer_atlas_notebook03_outputs.zip
  polymer_atlas_notebook02_outputs.zip


In [2]:
# ============================================================
# Extract project artifacts
# ============================================================

from pathlib import Path
import zipfile
import json
import numpy as np
import pandas as pd

BASE_DIR = Path("/content")

artifact_specs = {
    "nb02": {
        "zip": BASE_DIR / "polymer_atlas_notebook02_outputs.zip",
        "dir": BASE_DIR / "polymer_atlas_notebook02_outputs"
    },

    "nb03": {
        "zip": BASE_DIR / "polymer_atlas_notebook03_outputs.zip",
        "dir": BASE_DIR / "polymer_atlas_notebook03_outputs"
    },

    "nb04": {
        "zip": BASE_DIR / "polymer_atlas_notebook04_outputs.zip",
        "dir": BASE_DIR / "polymer_atlas_notebook04_outputs"
    }
}


for label, spec in artifact_specs.items():

    spec["dir"].mkdir(
        exist_ok=True
    )

    with zipfile.ZipFile(
        spec["zip"],
        "r"
    ) as z:

        z.extractall(
            spec["dir"]
        )

    print(
        f"{label.upper()} extracted -> "
        f"{spec['dir']}"
    )


NB02_DIR = artifact_specs["nb02"]["dir"]
NB03_DIR = artifact_specs["nb03"]["dir"]
NB04_DIR = artifact_specs["nb04"]["dir"]

NB02 extracted -> /content/polymer_atlas_notebook02_outputs
NB03 extracted -> /content/polymer_atlas_notebook03_outputs
NB04 extracted -> /content/polymer_atlas_notebook04_outputs


In [3]:
# ============================================================
# Load Notebook 02 representations and metadata
# ============================================================

X_interpretable = np.load(
    NB02_DIR / "X_interpretable.npy"
)

X_morgan = np.load(
    NB02_DIR / "X_morgan_count_r4.npy"
)

X_hybrid = np.load(
    NB02_DIR / "X_hybrid.npy"
)

metadata = pd.read_csv(
    NB02_DIR / "polymer_model_metadata.csv"
)


target_columns = [
    "Tg",
    "FFV",
    "Tc",
    "Density",
    "Rg"
]


print("NOTEBOOK 05 INPUT MATRICES")
print("=" * 75)

print(
    f"Metadata:       {metadata.shape}"
)

print(
    f"Interpretable:  {X_interpretable.shape}"
)

print(
    f"Morgan r4:      {X_morgan.shape}"
)

print(
    f"Hybrid:         {X_hybrid.shape}"
)

NOTEBOOK 05 INPUT MATRICES
Metadata:       (7973, 8)
Interpretable:  (7973, 36)
Morgan r4:      (7973, 2048)
Hybrid:         (7973, 2084)


In [4]:
# ============================================================
# Load prior notebook reference results
# ============================================================

nb03_holdout_results = pd.read_csv(
    NB03_DIR / "final_holdout_results.csv"
)

nb03_frozen_splits = pd.read_csv(
    NB03_DIR / "frozen_target_splits.csv"
)

nb04_similarity_summary = pd.read_csv(
    NB04_DIR / "chemical_similarity_summary.csv"
)

nb04_partial_correlations = pd.read_csv(
    NB04_DIR / "partial_similarity_error_correlations.csv"
)

nb04_bootstrap_summary = pd.read_csv(
    NB04_DIR / "bootstrap_performance_summary.csv"
)


print("PRIOR RESULTS LOADED")
print("=" * 75)

print(
    f"Notebook 03 holdout table: "
    f"{nb03_holdout_results.shape}"
)

print(
    f"Notebook 03 split table:   "
    f"{nb03_frozen_splits.shape}"
)

print(
    f"Notebook 04 similarity:    "
    f"{nb04_similarity_summary.shape}"
)

print(
    f"Notebook 04 bootstrap:     "
    f"{nb04_bootstrap_summary.shape}"
)

PRIOR RESULTS LOADED
Notebook 03 holdout table: (5, 8)
Notebook 03 split table:   (9505, 3)
Notebook 04 similarity:    (5, 15)
Notebook 04 bootstrap:     (10, 8)


## 5. Entry Integrity Audit

Notebook 05 combines structural representations, target labels, baseline predictions, and chemistry-aware reliability information produced in previous notebooks.

Before shared learning is attempted, the frozen artifacts are checked for:

- consistent polymer ordering;
- unique polymer identifiers;
- preserved target counts;
- finite numerical feature matrices;
- and expected target availability.

No model is trained until these conditions are confirmed.

In [5]:
# ============================================================
# Notebook 05 entry integrity audit
# ============================================================

n_polymers = len(metadata)

assert X_interpretable.shape[0] == n_polymers
assert X_morgan.shape[0] == n_polymers
assert X_hybrid.shape[0] == n_polymers

assert metadata["id"].is_unique

assert np.isfinite(
    X_interpretable
).all()

assert np.isfinite(
    X_morgan
).all()

assert np.isfinite(
    X_hybrid
).all()


target_counts = (
    metadata[target_columns]
    .notna()
    .sum()
)


print("NOTEBOOK 05 ENTRY AUDIT")
print("=" * 75)

print(
    f"Total polymers: {n_polymers:,}"
)

print("\nTarget availability:")

for target in target_columns:

    print(
        f"  {target:8s}: "
        f"{target_counts[target]:,}"
    )

print("\nFeature integrity:")
print("  Interpretable: finite")
print("  Morgan r4:     finite")
print("  Hybrid:        finite")

print("\nEntry audit: PASSED")

NOTEBOOK 05 ENTRY AUDIT
Total polymers: 7,973

Target availability:
  Tg      : 511
  FFV     : 7,030
  Tc      : 737
  Density : 613
  Rg      : 614

Feature integrity:
  Interpretable: finite
  Morgan r4:     finite
  Hybrid:        finite

Entry audit: PASSED


## 6. Target-Overlap Architecture

Multitask learning is scientifically justified only when targets share enough polymers or related structural information for useful transfer to occur.

Target availability is therefore treated as a graph in which:

- each target is a node;
- an edge represents polymers measured for both targets;
- and edge strength corresponds to the number of shared records.

Large overlaps provide direct opportunities for shared representation learning.

Small overlaps provide much weaker evidence that the targets should be learned jointly.

In [6]:
# ============================================================
# Pairwise target overlap
# ============================================================

overlap_matrix = pd.DataFrame(
    index=target_columns,
    columns=target_columns,
    dtype=int
)


for target_a in target_columns:

    for target_b in target_columns:

        overlap_matrix.loc[
            target_a,
            target_b
        ] = int(
            (
                metadata[target_a].notna()
                &
                metadata[target_b].notna()
            ).sum()
        )


print("PAIRWISE TARGET OVERLAP")
print("=" * 75)

display(
    overlap_matrix
)

PAIRWISE TARGET OVERLAP


,Tg,FFV,Tc,Density,Rg
Tg,511.0,1.0,32.0,24.0,24.0
FFV,1.0,7030.0,300.0,270.0,270.0
Tc,32.0,300.0,737.0,531.0,535.0
Density,24.0,270.0,531.0,613.0,610.0
Rg,24.0,270.0,535.0,610.0,614.0


In [7]:
# ============================================================
# Tc–Density–Rg shared-data block
# ============================================================

core_targets = [
    "Tc",
    "Density",
    "Rg"
]

core_complete_mask = (
    metadata[core_targets]
    .notna()
    .all(axis=1)
)

core_complete_idx = np.flatnonzero(
    core_complete_mask.to_numpy()
)


core_any_mask = (
    metadata[core_targets]
    .notna()
    .any(axis=1)
)

core_any_idx = np.flatnonzero(
    core_any_mask.to_numpy()
)


print("Tc–Density–Rg SHARED-LEARNING BLOCK")
print("=" * 75)

print(
    f"Complete triplets: "
    f"{len(core_complete_idx):,}"
)

print(
    f"At least one of the three targets: "
    f"{len(core_any_idx):,}"
)

print(
    "\nIndividual target counts:"
)

for target in core_targets:

    print(
        f"  {target:8s}: "
        f"{metadata[target].notna().sum():,}"
    )

Tc–Density–Rg SHARED-LEARNING BLOCK
Complete triplets: 531
At least one of the three targets: 819

Individual target counts:
  Tc      : 737
  Density : 613
  Rg      : 614


## 7. Property Relationships Within the Shared Measurement Block

Large record overlap does not automatically imply that targets should share a predictive representation.

The measured target values themselves are therefore examined within polymers for which Tc, Density, and Rg are all available.

Both Pearson and Spearman correlations are calculated.

Pearson correlation describes approximately linear association.

Spearman correlation describes monotonic association and is less sensitive to nonlinear scaling and extreme values.

These correlations are exploratory evidence for shared learning, not proof of causal physical relationships.

In [8]:
# ============================================================
# Target relationships on complete Tc–Density–Rg records
# ============================================================

from scipy.stats import (
    pearsonr,
    spearmanr
)

core_target_data = (
    metadata.loc[
        core_complete_idx,
        core_targets
    ]
    .copy()
)


pearson_matrix = (
    core_target_data
    .corr(
        method="pearson"
    )
)

spearman_matrix = (
    core_target_data
    .corr(
        method="spearman"
    )
)


print("PEARSON CORRELATION")
print("=" * 75)

display(
    pearson_matrix.round(4)
)


print("\nSPEARMAN CORRELATION")
print("=" * 75)

display(
    spearman_matrix.round(4)
)

PEARSON CORRELATION


,Tc,Density,Rg
Tc,1.0000,-0.4885,0.5530
Density,-0.4885,1.0000,0.0211
Rg,0.5530,0.0211,1.0000



SPEARMAN CORRELATION


,Tc,Density,Rg
Tc,1.0000,-0.4538,0.5030
Density,-0.4538,1.0000,0.1166
Rg,0.5030,0.1166,1.0000


In [9]:
relationship_rows = []

for i, target_a in enumerate(
    core_targets
):

    for target_b in core_targets[
        i + 1:
    ]:

        x = core_target_data[
            target_a
        ].to_numpy()

        y = core_target_data[
            target_b
        ].to_numpy()

        pearson_r, pearson_p = (
            pearsonr(
                x,
                y
            )
        )

        spearman_r, spearman_p = (
            spearmanr(
                x,
                y
            )
        )

        relationship_rows.append({
            "Target_A":
                target_a,

            "Target_B":
                target_b,

            "n":
                len(x),

            "Pearson_r":
                pearson_r,

            "Pearson_p":
                pearson_p,

            "Spearman_rho":
                spearman_r,

            "Spearman_p":
                spearman_p
        })


core_relationships = pd.DataFrame(
    relationship_rows
)


print(
    "Tc–Density–Rg PROPERTY RELATIONSHIPS"
)
print("=" * 100)

display(
    core_relationships.round(4)
)

Tc–Density–Rg PROPERTY RELATIONSHIPS


,Target_A,Target_B,n,Pearson_r,Pearson_p,Spearman_rho,Spearman_p
0,Tc,Density,531,-0.4885,0.0000,-0.4538,0.0000
1,Tc,Rg,531,0.5530,0.0000,0.5030,0.0000
2,Density,Rg,531,0.0211,0.6273,0.1166,0.0071


## 8. Missing-Label Structure in the Shared-Learning Block

A complete-case multitask model can use only polymers for which all three targets are available.

Masked multitask learning can potentially use additional polymers with only one or two measured targets.

The missing-label pattern is therefore quantified before choosing the final shared-learning architecture.

In [10]:
# ============================================================
# Missing-label patterns for Tc, Density, Rg
# ============================================================

core_availability = (
    metadata[core_targets]
    .notna()
    .astype(int)
)


core_availability[
    "pattern"
] = (
    core_availability[
        core_targets
    ]
    .astype(str)
    .agg(
        "".join,
        axis=1
    )
)


pattern_counts = (
    core_availability[
        "pattern"
    ]
    .value_counts()
    .sort_index()
)


pattern_labels = {
    "000": "none",
    "001": "Rg only",
    "010": "Density only",
    "011": "Density + Rg",
    "100": "Tc only",
    "101": "Tc + Rg",
    "110": "Tc + Density",
    "111": "Tc + Density + Rg"
}


missing_pattern_table = pd.DataFrame({
    "Pattern":
        pattern_counts.index,

    "Meaning":
        [
            pattern_labels.get(
                p,
                p
            )
            for p in pattern_counts.index
        ],

    "Count":
        pattern_counts.values
})


print(
    "Tc–Density–Rg LABEL-AVAILABILITY PATTERNS"
)
print("=" * 80)

display(
    missing_pattern_table
)

Tc–Density–Rg LABEL-AVAILABILITY PATTERNS


,Pattern,Meaning,Count
0,000,none,7154
1,010,Density only,3
2,011,Density + Rg,79
3,100,Tc only,202
4,101,Tc + Rg,4
5,111,Tc + Density + Rg,531


## 9. Shared-Learning Experimental Design

The target-overlap analysis identifies Tc, Density, and Rg as the appropriate shared-learning block.

A total of 819 polymers contain at least one of these three measurements, while 531 contain all three.

Using only complete triplets would therefore discard 288 polymers with scientifically useful partial labels.

Masked multitask learning is used so that each polymer contributes only to the target losses for which an experimental measurement is available.

The three properties do not exhibit identical statistical relationships.

Tc shows moderate association with both Density and Rg, whereas Density and Rg are only weakly related directly.

The multitask experiment therefore tests whether a shared structural representation produces positive transfer rather than assuming that all three properties must benefit from joint learning.

### Fair-comparison principle

A single polymer-level train/validation/test partition is used for all three targets.

This prevents a polymer used for testing one property from entering the shared training representation through another property's label.

Matched single-task and multitask neural networks use:

- the same polymer partitions;
- the same input representation;
- comparable hidden architectures;
- the same target standardization;
- the same early-stopping procedure.

Differences in performance can therefore be attributed primarily to the learning strategy.

### Input representation

The raw Hybrid representation contains 2,084 features, which is large relative to the 819-polymer shared-learning dataset.

A compact neural representation is therefore constructed from:

- the 36 interpretable polymer descriptors;
- a training-fitted truncated-SVD representation of the 2,048 Morgan-count features.

All dimensionality reduction and scaling parameters are fitted exclusively on the training partition.

In [11]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

core_union_idx = np.flatnonzero(
    metadata[core_targets]
    .notna()
    .any(axis=1)
    .to_numpy()
)

complete_indicator = (
    metadata.loc[
        core_union_idx,
        core_targets
    ]
    .notna()
    .all(axis=1)
    .astype(int)
    .to_numpy()
)

# 70% train, 30% temporary
train_idx, temp_idx, train_flag, temp_flag = (
    train_test_split(
        core_union_idx,
        complete_indicator,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=complete_indicator
    )
)

# Split temporary set equally:
# 15% validation, 15% test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_flag
)

print("GLOBAL SHARED-LEARNING SPLIT")
print("=" * 75)

print(
    f"Training:   {len(train_idx):,} "
    f"({100*len(train_idx)/len(core_union_idx):.1f}%)"
)

print(
    f"Validation: {len(val_idx):,} "
    f"({100*len(val_idx)/len(core_union_idx):.1f}%)"
)

print(
    f"Test:       {len(test_idx):,} "
    f"({100*len(test_idx)/len(core_union_idx):.1f}%)"
)

assert (
    set(train_idx).isdisjoint(val_idx)
    and set(train_idx).isdisjoint(test_idx)
    and set(val_idx).isdisjoint(test_idx)
)

print("\nPolymer-level leakage check: PASSED")

GLOBAL SHARED-LEARNING SPLIT
Training:   573 (70.0%)
Validation: 123 (15.0%)
Test:       123 (15.0%)

Polymer-level leakage check: PASSED


In [12]:
partition_indices = {
    "Train": train_idx,
    "Validation": val_idx,
    "Test": test_idx
}

partition_label_rows = []

for partition_name, idx in partition_indices.items():

    row = {
        "Partition": partition_name,
        "Polymers": len(idx)
    }

    for target in core_targets:

        row[target] = int(
            metadata.loc[
                idx,
                target
            ]
            .notna()
            .sum()
        )

    row["Complete_triplets"] = int(
        metadata.loc[
            idx,
            core_targets
        ]
        .notna()
        .all(axis=1)
        .sum()
    )

    partition_label_rows.append(row)


partition_label_counts = pd.DataFrame(
    partition_label_rows
)

print("LABEL AVAILABILITY BY PARTITION")
print("=" * 90)

display(
    partition_label_counts
)

LABEL AVAILABILITY BY PARTITION


,Partition,Polymers,Tc,Density,Rg,Complete_triplets
0,Train,573,518,427,428,372
1,Validation,123,108,95,95,80
2,Test,123,111,91,91,79


## 11. Compact Hybrid Representation for Neural Learning

The complete Hybrid representation contains 2,084 dimensions.

For the relatively small shared-learning dataset, directly fitting a neural network to this high-dimensional representation would increase model variance and overfitting risk.

The Morgan-count component is therefore compressed using Truncated Singular Value Decomposition.

Only the training partition is used to fit the dimensionality-reduction transformation.

The resulting neural representation contains:

$$
36 + 128 = 164
$$

features:

- 36 interpretable descriptors;
- 128 latent Morgan structural components.

The interpretable descriptors and latent Morgan components are subsequently standardized using statistics calculated only from the training partition.

In [13]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

N_MORGAN_COMPONENTS = 128

morgan_svd = TruncatedSVD(
    n_components=N_MORGAN_COMPONENTS,
    random_state=RANDOM_STATE
)

# Fit ONLY on training polymers
morgan_svd.fit(
    X_morgan[train_idx]
)

X_morgan_latent = morgan_svd.transform(
    X_morgan
)

X_neural_raw = np.hstack([
    X_interpretable,
    X_morgan_latent
])


feature_scaler = StandardScaler()

feature_scaler.fit(
    X_neural_raw[train_idx]
)

X_neural = feature_scaler.transform(
    X_neural_raw
).astype(np.float32)


print("NEURAL INPUT REPRESENTATION")
print("=" * 75)

print(
    f"Interpretable dimensions: "
    f"{X_interpretable.shape[1]}"
)

print(
    f"Morgan SVD dimensions:    "
    f"{N_MORGAN_COMPONENTS}"
)

print(
    f"Final neural dimensions:  "
    f"{X_neural.shape[1]}"
)

print(
    f"SVD variance captured:    "
    f"{morgan_svd.explained_variance_ratio_.sum():.4f}"
)

print(
    f"NaN values:               "
    f"{np.isnan(X_neural).sum()}"
)

print(
    f"Infinite values:          "
    f"{np.isinf(X_neural).sum()}"
)

NEURAL INPUT REPRESENTATION
Interpretable dimensions: 36
Morgan SVD dimensions:    128
Final neural dimensions:  164
SVD variance captured:    0.9759
NaN values:               0
Infinite values:          0


## 12. Target Standardization

Multitask losses must not be dominated by differences in numerical target scale.

Each property is standardized independently using the mean and standard deviation calculated from its available labels in the training partition:

$$
z =
\frac{y-\mu_{\mathrm{train}}}
{\sigma_{\mathrm{train}}}
$$

The same training-derived transformation is applied to validation and test labels.

Predictions are converted back to the original property scale before reporting R², MAE, and RMSE.

In [14]:
target_statistics = {}

Y_standardized = np.full(
    (
        len(metadata),
        len(core_targets)
    ),
    np.nan,
    dtype=np.float32
)

Y_mask = np.zeros(
    (
        len(metadata),
        len(core_targets)
    ),
    dtype=np.float32
)


for j, target in enumerate(core_targets):

    train_values = (
        metadata.loc[
            train_idx,
            target
        ]
        .dropna()
        .to_numpy(dtype=float)
    )

    mean_value = (
        train_values.mean()
    )

    std_value = (
        train_values.std(
            ddof=1
        )
    )

    target_statistics[target] = {
        "mean": mean_value,
        "std": std_value
    }

    available = (
        metadata[target]
        .notna()
        .to_numpy()
    )

    values = (
        metadata.loc[
            available,
            target
        ]
        .to_numpy(dtype=float)
    )

    Y_standardized[
        available,
        j
    ] = (
        (
            values
            - mean_value
        )
        / std_value
    )

    Y_mask[
        available,
        j
    ] = 1.0


target_statistics_table = pd.DataFrame([
    {
        "Target": target,
        "Train_mean":
            target_statistics[target]["mean"],
        "Train_std":
            target_statistics[target]["std"]
    }
    for target in core_targets
])


print("TRAINING-DERIVED TARGET STANDARDIZATION")
print("=" * 80)

display(
    target_statistics_table.round(4)
)

TRAINING-DERIVED TARGET STANDARDIZATION


,Target,Train_mean,Train_std
0,Tc,0.2574,0.0920
1,Density,0.9834,0.1512
2,Rg,16.4107,4.5711


In [15]:
import torch
import torch.nn as nn
from torch.utils.data import (
    Dataset,
    DataLoader
)

print("PYTORCH ENVIRONMENT")
print("=" * 75)

print(
    f"PyTorch version: "
    f"{torch.__version__}"
)

print(
    f"CUDA available:  "
    f"{torch.cuda.is_available()}"
)

if torch.cuda.is_available():

    device = torch.device(
        "cuda"
    )

    print(
        f"GPU:             "
        f"{torch.cuda.get_device_name(0)}"
    )

else:

    device = torch.device(
        "cpu"
    )

    print(
        "GPU unavailable — using CPU."
    )

print(
    f"Selected device: {device}"
)

PYTORCH ENVIRONMENT
PyTorch version: 2.11.0+cu128
CUDA available:  True
GPU:             Tesla T4
Selected device: cuda


In [16]:
class PolymerMultitaskDataset(Dataset):

    def __init__(
        self,
        indices,
        X,
        Y,
        mask
    ):

        self.indices = np.asarray(
            indices
        )

        self.X = torch.tensor(
            X[self.indices],
            dtype=torch.float32
        )

        self.Y = torch.tensor(
            np.nan_to_num(
                Y[self.indices],
                nan=0.0
            ),
            dtype=torch.float32
        )

        self.mask = torch.tensor(
            mask[self.indices],
            dtype=torch.float32
        )


    def __len__(self):

        return len(
            self.indices
        )


    def __getitem__(
        self,
        idx
    ):

        return (
            self.X[idx],
            self.Y[idx],
            self.mask[idx]
        )

In [17]:
BATCH_SIZE = 64

train_dataset = PolymerMultitaskDataset(
    train_idx,
    X_neural,
    Y_standardized,
    Y_mask
)

val_dataset = PolymerMultitaskDataset(
    val_idx,
    X_neural,
    Y_standardized,
    Y_mask
)

test_dataset = PolymerMultitaskDataset(
    test_idx,
    X_neural,
    Y_standardized,
    Y_mask
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


print("MULTITASK DATA LOADERS")
print("=" * 75)

print(
    f"Train batches:      "
    f"{len(train_loader)}"
)

print(
    f"Validation batches: "
    f"{len(val_loader)}"
)

print(
    f"Test batches:       "
    f"{len(test_loader)}"
)

MULTITASK DATA LOADERS
Train batches:      9
Validation batches: 2
Test batches:       2


## 15. Shared-Trunk Multitask Neural Network

The multitask architecture contains a shared nonlinear representation followed by separate property-specific prediction heads.

The shared pathway is:

$$
164
\rightarrow
256
\rightarrow
128
\rightarrow
64
$$

The 64-dimensional latent representation is then passed independently to three output heads:

$$
h_{\mathrm{shared}}
\rightarrow
\begin{cases}
\hat{T}_c \\
\widehat{\mathrm{Density}} \\
\hat{R}_g
\end{cases}
$$

Dropout and weight decay are used as regularization because the shared-learning dataset remains relatively small.

A masked loss ensures that missing labels do not contribute to optimization.

In [18]:
class SharedMultitaskNet(nn.Module):

    def __init__(
        self,
        input_dim
    ):

        super().__init__()

        self.shared = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                0.20
            ),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU(),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                128,
                64
            ),

            nn.ReLU()
        )

        self.heads = nn.ModuleList([
            nn.Linear(
                64,
                1
            )
            for _ in core_targets
        ])


    def forward(
        self,
        x
    ):

        shared = self.shared(
            x
        )

        outputs = [
            head(shared)
            for head in self.heads
        ]

        return torch.cat(
            outputs,
            dim=1
        )

## 16. Masked Multitask Objective

Not every polymer has measurements for all three targets.

A conventional multitask mean-squared-error loss would therefore either require discarding partially labelled polymers or incorrectly treating missing measurements as numerical targets.

Instead, a masked task-balanced loss is used.

For target $t$:

$$
L_t
=
\frac{
\sum_i m_{it}
\left(
y_{it}-\hat{y}_{it}
\right)^2
}{
\sum_i m_{it}
}
$$

where $m_{it}=1$ only when polymer $i$ has an observed value for target $t$.

The complete multitask objective is:

$$
L_{\mathrm{MT}}
=
\frac{1}{T}
\sum_{t=1}^{T}
L_t
$$

for targets represented in the current batch.

Because targets were standardized beforehand, each task contributes on a comparable numerical scale.

Averaging the loss separately by task also prevents Tc, which contains more labels, from dominating optimization simply because it has more observed measurements.

In [19]:
# ============================================================
# Task-balanced masked multitask MSE
# ============================================================

def masked_multitask_mse(
    predictions,
    targets,
    mask
):

    task_losses = []

    for j in range(
        predictions.shape[1]
    ):

        valid = (
            mask[:, j] > 0
        )

        if valid.any():

            task_loss = (
                (
                    predictions[
                        valid,
                        j
                    ]
                    -
                    targets[
                        valid,
                        j
                    ]
                ) ** 2
            ).mean()

            task_losses.append(
                task_loss
            )

    if len(task_losses) == 0:
        raise RuntimeError(
            "Batch contains no observed labels."
        )

    return torch.stack(
        task_losses
    ).mean()


print(
    "Masked task-balanced loss defined."
)

Masked task-balanced loss defined.


## 17. Matched Single-Task Neural Baselines

Positive transfer cannot be established by comparing the multitask neural network only with the classical models from Notebook 03.

The multitask architecture is therefore compared with matched single-task neural networks.

Each single-task network uses:

- the same 164-dimensional input representation;
- the same hidden layer dimensions;
- the same activation functions;
- the same dropout regularization;
- the same optimizer and early-stopping strategy.

The single-task models differ only in that their hidden representation is learned from one target rather than jointly from Tc, Density, and Rg.

This provides a controlled test of whether parameter sharing itself is beneficial.

In [20]:
class SingleTaskNet(nn.Module):

    def __init__(
        self,
        input_dim
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                0.20
            ),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU(),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                128,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                1
            )
        )


    def forward(
        self,
        x
    ):

        return self.network(
            x
        )

## 18. Neural Training Protocol

Neural-network performance can vary because of random weight initialization, mini-batch ordering, and stochastic optimization.

The shared-learning experiment is therefore repeated across five independent random seeds.

No hyperparameter search is conducted.

The neural architecture and optimization settings are frozen before observing test performance.

Training uses:

- AdamW optimization;
- learning rate = 0.001;
- weight decay = 0.0001;
- maximum 500 epochs;
- early stopping based exclusively on validation loss;
- patience = 40 epochs.

The test partition is never used for early stopping or model selection.

For each seed, the model state corresponding to the lowest validation loss is restored before test evaluation.

In [21]:
import random
import copy

NEURAL_SEEDS = [
    42,
    52,
    62,
    72,
    82
]

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 500
PATIENCE = 40


def set_all_seeds(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [22]:
def build_multitask_loaders(
    seed
):

    generator = torch.Generator()

    generator.manual_seed(
        seed
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    return (
        train_loader,
        val_loader,
        test_loader
    )

In [23]:
def evaluate_multitask_loss(
    model,
    loader
):

    model.eval()

    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():

        for X_batch, y_batch, mask_batch in loader:

            X_batch = X_batch.to(
                device
            )

            y_batch = y_batch.to(
                device
            )

            mask_batch = mask_batch.to(
                device
            )

            predictions = model(
                X_batch
            )

            loss = masked_multitask_mse(
                predictions,
                y_batch,
                mask_batch
            )

            total_loss += (
                loss.item()
            )

            n_batches += 1

    return (
        total_loss
        / n_batches
    )

In [24]:
def train_multitask_model(
    seed
):

    set_all_seeds(
        seed
    )

    (
        train_loader_seed,
        val_loader_seed,
        test_loader_seed
    ) = build_multitask_loaders(
        seed
    )

    model = SharedMultitaskNet(
        input_dim=X_neural.shape[1]
    ).to(
        device
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    best_val_loss = np.inf
    best_state = None

    patience_counter = 0

    history = []

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        model.train()

        train_loss_sum = 0.0
        train_batches = 0

        for (
            X_batch,
            y_batch,
            mask_batch
        ) in train_loader_seed:

            X_batch = X_batch.to(
                device
            )

            y_batch = y_batch.to(
                device
            )

            mask_batch = mask_batch.to(
                device
            )

            optimizer.zero_grad()

            predictions = model(
                X_batch
            )

            loss = masked_multitask_mse(
                predictions,
                y_batch,
                mask_batch
            )

            loss.backward()

            optimizer.step()

            train_loss_sum += (
                loss.item()
            )

            train_batches += 1

        train_loss = (
            train_loss_sum
            / train_batches
        )

        val_loss = evaluate_multitask_loss(
            model,
            val_loader_seed
        )

        history.append({
            "Epoch":
                epoch,

            "Train_loss":
                train_loss,

            "Validation_loss":
                val_loss
        })

        if (
            val_loss
            <
            best_val_loss - 1e-6
        ):

            best_val_loss = (
                val_loss
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1

        if (
            patience_counter
            >= PATIENCE
        ):
            break

    model.load_state_dict(
        best_state
    )

    history = pd.DataFrame(
        history
    )

    return (
        model,
        history,
        test_loader_seed
    )

In [25]:
from torch.utils.data import TensorDataset


def build_single_task_loaders(
    target,
    seed
):

    target_j = (
        core_targets.index(
            target
        )
    )

    train_available = (
        metadata.loc[
            train_idx,
            target
        ]
        .notna()
        .to_numpy()
    )

    val_available = (
        metadata.loc[
            val_idx,
            target
        ]
        .notna()
        .to_numpy()
    )

    test_available = (
        metadata.loc[
            test_idx,
            target
        ]
        .notna()
        .to_numpy()
    )


    target_train_idx = (
        train_idx[
            train_available
        ]
    )

    target_val_idx = (
        val_idx[
            val_available
        ]
    )

    target_test_idx = (
        test_idx[
            test_available
        ]
    )


    train_single = TensorDataset(

        torch.tensor(
            X_neural[
                target_train_idx
            ],
            dtype=torch.float32
        ),

        torch.tensor(
            Y_standardized[
                target_train_idx,
                target_j
            ],
            dtype=torch.float32
        )
        .unsqueeze(1)
    )


    val_single = TensorDataset(

        torch.tensor(
            X_neural[
                target_val_idx
            ],
            dtype=torch.float32
        ),

        torch.tensor(
            Y_standardized[
                target_val_idx,
                target_j
            ],
            dtype=torch.float32
        )
        .unsqueeze(1)
    )


    test_single = TensorDataset(

        torch.tensor(
            X_neural[
                target_test_idx
            ],
            dtype=torch.float32
        ),

        torch.tensor(
            Y_standardized[
                target_test_idx,
                target_j
            ],
            dtype=torch.float32
        )
        .unsqueeze(1)
    )


    generator = torch.Generator()

    generator.manual_seed(
        seed
    )


    train_loader = DataLoader(
        train_single,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_single,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        test_single,
        batch_size=BATCH_SIZE,
        shuffle=False
    )


    return (
        train_loader,
        val_loader,
        test_loader,
        target_test_idx
    )

In [26]:
def evaluate_single_task_loss(
    model,
    loader
):

    model.eval()

    losses = []

    criterion = nn.MSELoss()

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(
                device
            )

            y_batch = y_batch.to(
                device
            )

            predictions = model(
                X_batch
            )

            loss = criterion(
                predictions,
                y_batch
            )

            losses.append(
                loss.item()
            )

    return np.mean(
        losses
    )

In [27]:
def train_single_task_model(
    target,
    seed
):

    set_all_seeds(
        seed
    )

    (
        train_loader,
        val_loader,
        test_loader,
        target_test_idx
    ) = build_single_task_loaders(
        target,
        seed
    )

    model = SingleTaskNet(
        input_dim=X_neural.shape[1]
    ).to(
        device
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    criterion = nn.MSELoss()

    best_val_loss = np.inf
    best_state = None

    patience_counter = 0

    history = []

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        model.train()

        train_losses = []

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(
                device
            )

            y_batch = y_batch.to(
                device
            )

            optimizer.zero_grad()

            predictions = model(
                X_batch
            )

            loss = criterion(
                predictions,
                y_batch
            )

            loss.backward()

            optimizer.step()

            train_losses.append(
                loss.item()
            )

        train_loss = np.mean(
            train_losses
        )

        val_loss = evaluate_single_task_loss(
            model,
            val_loader
        )

        history.append({
            "Epoch":
                epoch,

            "Train_loss":
                train_loss,

            "Validation_loss":
                val_loss
        })

        if (
            val_loss
            <
            best_val_loss - 1e-6
        ):

            best_val_loss = (
                val_loss
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1

        if (
            patience_counter
            >= PATIENCE
        ):
            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(
            history
        ),
        test_loader,
        target_test_idx
    )

## 22. Transfer Evaluation

Predictions are converted from standardized neural outputs back to the original physical property scale.

For each target, single-task and multitask models are evaluated on exactly the same polymer-level test partition.

Performance is reported using:

- R²;
- MAE;
- RMSE.

The transfer effect is quantified as:

$$
\Delta R^2
=
R^2_{\mathrm{MT}}
-
R^2_{\mathrm{ST}}
$$

and

$$
\Delta \mathrm{MAE}
=
\mathrm{MAE}_{\mathrm{MT}}
-
\mathrm{MAE}_{\mathrm{ST}}
$$

Positive $\Delta R^2$ and negative $\Delta$MAE indicate positive transfer.

Because neural optimization is stochastic, conclusions are based on results across multiple random seeds rather than a single training run.

In [28]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)


def original_scale_metrics(
    target,
    y_true_standardized,
    y_pred_standardized
):

    mean_value = (
        target_statistics[
            target
        ]["mean"]
    )

    std_value = (
        target_statistics[
            target
        ]["std"]
    )

    y_true = (
        y_true_standardized
        * std_value
        + mean_value
    )

    y_pred = (
        y_pred_standardized
        * std_value
        + mean_value
    )

    return {
        "R2":
            r2_score(
                y_true,
                y_pred
            ),

        "MAE":
            mean_absolute_error(
                y_true,
                y_pred
            ),

        "RMSE":
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred
                )
            )
    }

In [29]:
transfer_results = []

trained_multitask_models = {}
trained_single_task_models = {}

multitask_histories = {}
single_task_histories = {}


for seed in NEURAL_SEEDS:

    print(
        "\n"
        + "=" * 80
    )

    print(
        f"SEED {seed}"
    )

    print(
        "=" * 80
    )


    # ========================================================
    # MULTITASK MODEL
    # ========================================================

    print(
        "\nTraining multitask model..."
    )

    (
        mt_model,
        mt_history,
        _
    ) = train_multitask_model(
        seed
    )

    trained_multitask_models[
        seed
    ] = mt_model

    multitask_histories[
        seed
    ] = mt_history


    # Predict entire global test partition once
    mt_model.eval()

    with torch.no_grad():

        X_test_tensor = torch.tensor(
            X_neural[
                test_idx
            ],
            dtype=torch.float32
        ).to(
            device
        )

        mt_pred_all = (
            mt_model(
                X_test_tensor
            )
            .cpu()
            .numpy()
        )


    for target_j, target in enumerate(
        core_targets
    ):

        available = (
            metadata.loc[
                test_idx,
                target
            ]
            .notna()
            .to_numpy()
        )

        target_test_idx = (
            test_idx[
                available
            ]
        )

        y_true_std = (
            Y_standardized[
                target_test_idx,
                target_j
            ]
        )

        y_pred_std = (
            mt_pred_all[
                available,
                target_j
            ]
        )

        metrics = original_scale_metrics(
            target,
            y_true_std,
            y_pred_std
        )

        transfer_results.append({
            "Seed":
                seed,

            "Learning":
                "Multitask",

            "Target":
                target,

            "Test_n":
                len(
                    target_test_idx
                ),

            **metrics
        })


    # ========================================================
    # MATCHED SINGLE-TASK MODELS
    # ========================================================

    for target in core_targets:

        print(
            f"Training single-task {target}..."
        )

        (
            st_model,
            st_history,
            st_test_loader,
            target_test_idx
        ) = train_single_task_model(
            target,
            seed
        )

        trained_single_task_models[
            (
                seed,
                target
            )
        ] = st_model

        single_task_histories[
            (
                seed,
                target
            )
        ] = st_history


        predictions = []
        true_values = []

        st_model.eval()

        with torch.no_grad():

            for (
                X_batch,
                y_batch
            ) in st_test_loader:

                X_batch = X_batch.to(
                    device
                )

                pred = (
                    st_model(
                        X_batch
                    )
                    .cpu()
                    .numpy()
                    .ravel()
                )

                predictions.extend(
                    pred
                )

                true_values.extend(
                    y_batch
                    .numpy()
                    .ravel()
                )


        metrics = original_scale_metrics(
            target,
            np.asarray(
                true_values
            ),
            np.asarray(
                predictions
            )
        )

        transfer_results.append({
            "Seed":
                seed,

            "Learning":
                "SingleTask",

            "Target":
                target,

            "Test_n":
                len(
                    target_test_idx
                ),

            **metrics
        })


    print(
        f"\nSeed {seed} complete."
    )


SEED 42

Training multitask model...
Training single-task Tc...
Training single-task Density...
Training single-task Rg...

Seed 42 complete.

SEED 52

Training multitask model...
Training single-task Tc...
Training single-task Density...
Training single-task Rg...

Seed 52 complete.

SEED 62

Training multitask model...
Training single-task Tc...
Training single-task Density...
Training single-task Rg...

Seed 62 complete.

SEED 72

Training multitask model...
Training single-task Tc...
Training single-task Density...
Training single-task Rg...

Seed 72 complete.

SEED 82

Training multitask model...
Training single-task Tc...
Training single-task Density...
Training single-task Rg...

Seed 82 complete.


In [30]:
transfer_results = pd.DataFrame(
    transfer_results
)

print("ALL NEURAL TRANSFER RESULTS")
print("=" * 100)

display(
    transfer_results.round(4)
)

ALL NEURAL TRANSFER RESULTS


,Seed,Learning,Target,Test_n,R2,MAE,RMSE
0,42,Multitask,Tc,111,0.7602,0.0284,0.0408
1,42,Multitask,Density,91,0.5613,0.0382,0.0904
2,42,Multitask,Rg,91,0.7304,1.6399,2.5482
3,42,SingleTask,Tc,111,0.7668,0.0273,0.0403
4,42,SingleTask,Density,91,0.4108,0.0410,0.1048
5,42,SingleTask,Rg,91,0.7032,1.7250,2.6737
6,52,Multitask,Tc,111,0.7700,0.0272,0.0400
7,52,Multitask,Density,91,0.5677,0.0391,0.0898
8,52,Multitask,Rg,91,0.7218,1.6708,2.5884
9,52,SingleTask,Tc,111,0.7785,0.0274,0.0393


In [31]:
transfer_summary = (
    transfer_results
    .groupby(
        [
            "Target",
            "Learning"
        ],
        as_index=False
    )
    .agg(
        R2_mean=(
            "R2",
            "mean"
        ),

        R2_std=(
            "R2",
            "std"
        ),

        MAE_mean=(
            "MAE",
            "mean"
        ),

        MAE_std=(
            "MAE",
            "std"
        ),

        RMSE_mean=(
            "RMSE",
            "mean"
        ),

        RMSE_std=(
            "RMSE",
            "std"
        )
    )
)


print(
    "MULTI-SEED SINGLE-TASK VS MULTITASK SUMMARY"
)
print("=" * 110)

display(
    transfer_summary.round(4)
)

MULTI-SEED SINGLE-TASK VS MULTITASK SUMMARY


,Target,Learning,R2_mean,R2_std,MAE_mean,MAE_std,RMSE_mean,RMSE_std
0,Density,Multitask,0.5425,0.0459,0.0390,0.0015,0.0922,0.0046
1,Density,SingleTask,0.5298,0.0835,0.0386,0.0024,0.0933,0.0082
2,Rg,Multitask,0.7264,0.0146,1.6485,0.0347,2.5664,0.0693
3,Rg,SingleTask,0.7177,0.0094,1.6869,0.0390,2.6074,0.0433
4,Tc,Multitask,0.7739,0.0097,0.0278,0.0007,0.0396,0.0009
5,Tc,SingleTask,0.7728,0.0124,0.0280,0.0008,0.0397,0.0011


In [32]:
single_summary = (
    transfer_summary[
        transfer_summary[
            "Learning"
        ]
        == "SingleTask"
    ]
    .set_index(
        "Target"
    )
)


multi_summary = (
    transfer_summary[
        transfer_summary[
            "Learning"
        ]
        == "Multitask"
    ]
    .set_index(
        "Target"
    )
)


transfer_effect_rows = []

for target in core_targets:

    st = single_summary.loc[
        target
    ]

    mt = multi_summary.loc[
        target
    ]

    transfer_effect_rows.append({
        "Target":
            target,

        "SingleTask_R2":
            st["R2_mean"],

        "Multitask_R2":
            mt["R2_mean"],

        "Delta_R2":
            (
                mt["R2_mean"]
                -
                st["R2_mean"]
            ),

        "SingleTask_MAE":
            st["MAE_mean"],

        "Multitask_MAE":
            mt["MAE_mean"],

        "Delta_MAE":
            (
                mt["MAE_mean"]
                -
                st["MAE_mean"]
            ),

        "SingleTask_RMSE":
            st["RMSE_mean"],

        "Multitask_RMSE":
            mt["RMSE_mean"],

        "Delta_RMSE":
            (
                mt["RMSE_mean"]
                -
                st["RMSE_mean"]
            )
    })


transfer_effect = pd.DataFrame(
    transfer_effect_rows
)


print("MULTITASK TRANSFER EFFECT")
print("=" * 110)

display(
    transfer_effect.round(4)
)

MULTITASK TRANSFER EFFECT


,Target,SingleTask_R2,Multitask_R2,Delta_R2,SingleTask_MAE,Multitask_MAE,Delta_MAE,SingleTask_RMSE,Multitask_RMSE,Delta_RMSE
0,Tc,0.7728,0.7739,0.0012,0.0280,0.0278,-0.0002,0.0397,0.0396,-0.0001
1,Density,0.5298,0.5425,0.0127,0.0386,0.0390,0.0004,0.0933,0.0922,-0.0011
2,Rg,0.7177,0.7264,0.0087,1.6869,1.6485,-0.0384,2.6074,2.5664,-0.0410


## 25. Paired Transfer Robustness Across Random Seeds

Average performance can obscure whether multitask gains are reproducible across stochastic neural-network runs.

Because the single-task and multitask experiments used the same five random seeds and exactly the same polymer-level test partitions, transfer effects can be compared within each seed.

For every target and seed:

$$
\Delta R^2_s
=
R^2_{\mathrm{MT},s}
-
R^2_{\mathrm{ST},s}
$$

$$
\Delta \mathrm{MAE}_s
=
\mathrm{MAE}_{\mathrm{MT},s}
-
\mathrm{MAE}_{\mathrm{ST},s}
$$

$$
\Delta \mathrm{RMSE}_s
=
\mathrm{RMSE}_{\mathrm{MT},s}
-
\mathrm{RMSE}_{\mathrm{ST},s}
$$

Positive $\Delta R^2$ and negative error differences indicate positive transfer.

Because only five seeds are available, these paired results are interpreted descriptively rather than as high-powered inferential statistics.

In [33]:
# ============================================================
# Paired seed-level transfer analysis
# ============================================================

single_seed = (
    transfer_results[
        transfer_results["Learning"]
        == "SingleTask"
    ]
    .set_index(
        ["Seed", "Target"]
    )
)

multi_seed = (
    transfer_results[
        transfer_results["Learning"]
        == "Multitask"
    ]
    .set_index(
        ["Seed", "Target"]
    )
)


paired_transfer_rows = []

for seed in NEURAL_SEEDS:

    for target in core_targets:

        st = single_seed.loc[
            (seed, target)
        ]

        mt = multi_seed.loc[
            (seed, target)
        ]

        paired_transfer_rows.append({
            "Seed": seed,
            "Target": target,

            "SingleTask_R2":
                st["R2"],

            "Multitask_R2":
                mt["R2"],

            "Delta_R2":
                mt["R2"] - st["R2"],

            "Delta_MAE":
                mt["MAE"] - st["MAE"],

            "Delta_RMSE":
                mt["RMSE"] - st["RMSE"]
        })


paired_transfer = pd.DataFrame(
    paired_transfer_rows
)

print("PAIRED SEED-LEVEL TRANSFER")
print("=" * 100)

display(
    paired_transfer.round(4)
)

PAIRED SEED-LEVEL TRANSFER


,Seed,Target,SingleTask_R2,Multitask_R2,Delta_R2,Delta_MAE,Delta_RMSE
0,42,Tc,0.7668,0.7602,-0.0065,0.0011,0.0006
1,42,Density,0.4108,0.5613,0.1505,-0.0028,-0.0144
2,42,Rg,0.7032,0.7304,0.0272,-0.0850,-0.1255
3,52,Tc,0.7785,0.7700,-0.0085,-0.0002,0.0007
4,52,Density,0.4756,0.5677,0.0921,-0.0011,-0.0091
5,52,Rg,0.7263,0.7218,-0.0044,0.0208,0.0207
6,62,Tc,0.7644,0.7782,0.0138,-0.0002,-0.0012
7,62,Density,0.5874,0.4922,-0.0951,0.0038,0.0096
8,62,Rg,0.7219,0.7136,-0.0083,0.0509,0.0383
9,72,Tc,0.7920,0.7863,-0.0056,-0.0009,0.0005


In [34]:
paired_transfer_summary = (
    paired_transfer
    .groupby(
        "Target",
        as_index=False
    )
    .agg(
        Mean_Delta_R2=(
            "Delta_R2",
            "mean"
        ),

        Median_Delta_R2=(
            "Delta_R2",
            "median"
        ),

        Std_Delta_R2=(
            "Delta_R2",
            "std"
        ),

        Mean_Delta_MAE=(
            "Delta_MAE",
            "mean"
        ),

        Mean_Delta_RMSE=(
            "Delta_RMSE",
            "mean"
        )
    )
)


win_counts = (
    paired_transfer
    .assign(
        R2_MT_win=lambda df:
            df["Delta_R2"] > 0,

        MAE_MT_win=lambda df:
            df["Delta_MAE"] < 0,

        RMSE_MT_win=lambda df:
            df["Delta_RMSE"] < 0
    )
    .groupby(
        "Target",
        as_index=False
    )
    .agg(
        R2_wins=(
            "R2_MT_win",
            "sum"
        ),

        MAE_wins=(
            "MAE_MT_win",
            "sum"
        ),

        RMSE_wins=(
            "RMSE_MT_win",
            "sum"
        )
    )
)


paired_transfer_summary = (
    paired_transfer_summary
    .merge(
        win_counts,
        on="Target"
    )
)


print(
    "PAIRED TRANSFER ROBUSTNESS SUMMARY"
)
print("=" * 110)

display(
    paired_transfer_summary.round(4)
)

PAIRED TRANSFER ROBUSTNESS SUMMARY


,Target,Mean_Delta_R2,Median_Delta_R2,Std_Delta_R2,Mean_Delta_MAE,Mean_Delta_RMSE,R2_wins,MAE_wins,RMSE_wins
0,Density,0.0127,-0.0124,0.1056,0.0004,-0.0011,2,2,2
1,Rg,0.0087,-0.0044,0.0214,-0.0384,-0.0410,2,3,2
2,Tc,0.0012,-0.0056,0.0111,-0.0002,-0.0001,2,4,2


## 26. Neural Training-Stability Audit

The transfer comparison is meaningful only if both single-task and multitask models trained normally.

The stored optimization histories are therefore examined to determine:

- the number of epochs completed before early stopping;
- the minimum validation loss reached;
- whether models consistently converged before the maximum epoch limit;
- and whether unusual transfer results correspond to unstable optimization.

This diagnostic does not modify any trained model.

In [35]:
# ============================================================
# Training stability summary
# ============================================================

training_stability_rows = []


# Multitask histories
for seed in NEURAL_SEEDS:

    history = multitask_histories[seed]

    best_row = history.loc[
        history["Validation_loss"].idxmin()
    ]

    training_stability_rows.append({
        "Seed": seed,
        "Learning": "Multitask",
        "Target": "Shared",
        "Epochs_run": len(history),
        "Best_epoch": int(
            best_row["Epoch"]
        ),
        "Best_validation_loss":
            best_row["Validation_loss"]
    })


# Single-task histories
for seed in NEURAL_SEEDS:

    for target in core_targets:

        history = single_task_histories[
            (seed, target)
        ]

        best_row = history.loc[
            history[
                "Validation_loss"
            ].idxmin()
        ]

        training_stability_rows.append({
            "Seed": seed,
            "Learning": "SingleTask",
            "Target": target,
            "Epochs_run": len(history),
            "Best_epoch": int(
                best_row["Epoch"]
            ),
            "Best_validation_loss":
                best_row[
                    "Validation_loss"
                ]
        })


training_stability = pd.DataFrame(
    training_stability_rows
)

print("NEURAL TRAINING-STABILITY AUDIT")
print("=" * 100)

display(
    training_stability.round(4)
)

NEURAL TRAINING-STABILITY AUDIT


,Seed,Learning,Target,Epochs_run,Best_epoch,Best_validation_loss
0,42,Multitask,Shared,63,23,0.1489
1,52,Multitask,Shared,82,42,0.1585
2,62,Multitask,Shared,83,43,0.1466
3,72,Multitask,Shared,172,132,0.1528
4,82,Multitask,Shared,65,25,0.1516
5,42,SingleTask,Tc,92,52,0.1395
6,42,SingleTask,Density,104,64,0.0752
7,42,SingleTask,Rg,61,21,0.2172
8,52,SingleTask,Tc,59,19,0.1337
9,52,SingleTask,Density,53,13,0.0770


In [36]:
training_stability_summary = (
    training_stability
    .groupby(
        ["Learning", "Target"],
        as_index=False
    )
    .agg(
        Mean_epochs_run=(
            "Epochs_run",
            "mean"
        ),

        Min_epochs_run=(
            "Epochs_run",
            "min"
        ),

        Max_epochs_run=(
            "Epochs_run",
            "max"
        ),

        Mean_best_epoch=(
            "Best_epoch",
            "mean"
        )
    )
)

print("TRAINING-STABILITY SUMMARY")
print("=" * 90)

display(
    training_stability_summary.round(2)
)

TRAINING-STABILITY SUMMARY


,Learning,Target,Mean_epochs_run,Min_epochs_run,Max_epochs_run,Mean_best_epoch
0,Multitask,Shared,93.0,63,172,53.0
1,SingleTask,Density,95.6,53,138,55.6
2,SingleTask,Rg,60.2,57,65,20.2
3,SingleTask,Tc,69.8,50,92,29.8


## 27. Complete-Case Multitask Ablation

Masked multitask learning allows partially labelled polymers to contribute to training.

To determine whether these additional records materially affect predictive performance, a complete-case multitask control is trained using only polymers with observed Tc, Density, and Rg values.

The architecture, optimization procedure, global test partition, and random seeds are unchanged.

The comparison therefore distinguishes:

- shared learning using all available partial labels;
- shared learning restricted to complete target triplets.

If masked learning performs better, the additional partially labelled polymers provide useful information.

If performance is similar, most transferable information is already contained within the complete-triplet subset.

In [37]:
complete_train_idx = train_idx[
    metadata.loc[
        train_idx,
        core_targets
    ]
    .notna()
    .all(axis=1)
    .to_numpy()
]

complete_val_idx = val_idx[
    metadata.loc[
        val_idx,
        core_targets
    ]
    .notna()
    .all(axis=1)
    .to_numpy()
]


print("COMPLETE-CASE MULTITASK DATA")
print("=" * 75)

print(
    f"Complete training polymers:   "
    f"{len(complete_train_idx)}"
)

print(
    f"Complete validation polymers: "
    f"{len(complete_val_idx)}"
)

print(
    f"Masked-training polymers:     "
    f"{len(train_idx)}"
)

COMPLETE-CASE MULTITASK DATA
Complete training polymers:   372
Complete validation polymers: 80
Masked-training polymers:     573


In [38]:
def build_complete_case_loaders(
    seed
):

    train_complete_dataset = (
        PolymerMultitaskDataset(
            complete_train_idx,
            X_neural,
            Y_standardized,
            Y_mask
        )
    )

    val_complete_dataset = (
        PolymerMultitaskDataset(
            complete_val_idx,
            X_neural,
            Y_standardized,
            Y_mask
        )
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_complete_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_complete_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    return (
        train_loader,
        val_loader
    )

In [39]:
def train_complete_case_multitask(
    seed
):

    set_all_seeds(seed)

    (
        train_loader,
        val_loader
    ) = build_complete_case_loaders(
        seed
    )

    model = SharedMultitaskNet(
        input_dim=X_neural.shape[1]
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    best_val_loss = np.inf
    best_state = None
    patience_counter = 0

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        model.train()

        for (
            X_batch,
            y_batch,
            mask_batch
        ) in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)

            optimizer.zero_grad()

            predictions = model(
                X_batch
            )

            loss = masked_multitask_mse(
                predictions,
                y_batch,
                mask_batch
            )

            loss.backward()
            optimizer.step()

        val_loss = evaluate_multitask_loss(
            model,
            val_loader
        )

        if val_loss < (
            best_val_loss - 1e-6
        ):

            best_val_loss = val_loss

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    model.load_state_dict(
        best_state
    )

    return model

In [40]:
complete_case_results = []

for seed in NEURAL_SEEDS:

    print(
        f"Complete-case multitask seed "
        f"{seed}..."
    )

    model = train_complete_case_multitask(
        seed
    )

    model.eval()

    with torch.no_grad():

        X_test_tensor = torch.tensor(
            X_neural[test_idx],
            dtype=torch.float32
        ).to(device)

        predictions = (
            model(
                X_test_tensor
            )
            .cpu()
            .numpy()
        )


    for j, target in enumerate(
        core_targets
    ):

        available = (
            metadata.loc[
                test_idx,
                target
            ]
            .notna()
            .to_numpy()
        )

        target_test_idx = (
            test_idx[
                available
            ]
        )

        y_true_std = (
            Y_standardized[
                target_test_idx,
                j
            ]
        )

        y_pred_std = (
            predictions[
                available,
                j
            ]
        )

        metrics = original_scale_metrics(
            target,
            y_true_std,
            y_pred_std
        )

        complete_case_results.append({
            "Seed": seed,
            "Target": target,
            "Learning":
                "CompleteCaseMultitask",
            **metrics
        })


complete_case_results = pd.DataFrame(
    complete_case_results
)

Complete-case multitask seed 42...
Complete-case multitask seed 52...
Complete-case multitask seed 62...
Complete-case multitask seed 72...
Complete-case multitask seed 82...


In [41]:
three_way_transfer_results = pd.concat(
    [
        transfer_results,
        complete_case_results
    ],
    ignore_index=True
)


three_way_summary = (
    three_way_transfer_results
    .groupby(
        ["Target", "Learning"],
        as_index=False
    )
    .agg(
        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
        MAE_mean=("MAE", "mean"),
        RMSE_mean=("RMSE", "mean")
    )
)


print(
    "SINGLE-TASK VS COMPLETE-CASE "
    "VS MASKED MULTITASK"
)
print("=" * 115)

display(
    three_way_summary.round(4)
)

SINGLE-TASK VS COMPLETE-CASE VS MASKED MULTITASK


,Target,Learning,R2_mean,R2_std,MAE_mean,RMSE_mean
0,Density,CompleteCaseMultitask,0.5075,0.0361,0.0425,0.0958
1,Density,Multitask,0.5425,0.0459,0.0390,0.0922
2,Density,SingleTask,0.5298,0.0835,0.0386,0.0933
3,Rg,CompleteCaseMultitask,0.6840,0.0218,1.7838,2.7576
4,Rg,Multitask,0.7264,0.0146,1.6485,2.5664
5,Rg,SingleTask,0.7177,0.0094,1.6869,2.6074
6,Tc,CompleteCaseMultitask,0.7562,0.0115,0.0290,0.0412
7,Tc,Multitask,0.7739,0.0097,0.0278,0.0396
8,Tc,SingleTask,0.7728,0.0124,0.0280,0.0397


# Part B — Predictive Reliability

## 28. Reliability Experimental Design

Part A showed that masked multitask learning can exploit partially labelled polymers efficiently, although its predictive advantage over matched single-task learning is modest and seed dependent.

Notebook 04 also showed that chemical similarity is an informative reliability indicator for some properties but not universally.

The next objective is therefore:

> **Can prediction-level uncertainty identify difficult polymer predictions directly?**

A five-member multitask neural ensemble is used to estimate prediction disagreement.

However, calibrated prediction intervals require a calibration dataset that is independent of both model training and early stopping.

A new polymer-level four-way partition is therefore constructed:

$$
65\% \text{ training}
+
15\% \text{ validation}
+
10\% \text{ calibration}
+
10\% \text{ reliability test}
$$

The roles of these subsets are strictly separated:

- **training:** neural-network fitting;
- **validation:** early stopping;
- **calibration:** conformal interval calibration;
- **test:** final reliability evaluation.

The architecture, representation size, optimizer, regularization, and training protocol are frozen from Part A.

No new neural hyperparameter optimization is performed.

In [42]:
# ============================================================
# Four-way polymer split for reliability analysis
# ============================================================

RELIABILITY_RANDOM_STATE = 314

reliability_union_idx = core_union_idx.copy()

reliability_complete_flag = (
    metadata.loc[
        reliability_union_idx,
        core_targets
    ]
    .notna()
    .all(axis=1)
    .astype(int)
    .to_numpy()
)

# 65% train, 35% remaining
(
    rel_train_idx,
    rel_remaining_idx,
    rel_train_flag,
    rel_remaining_flag
) = train_test_split(
    reliability_union_idx,
    reliability_complete_flag,
    test_size=0.35,
    random_state=RELIABILITY_RANDOM_STATE,
    stratify=reliability_complete_flag
)

# 35% -> 15% validation + 20% remainder
(
    rel_val_idx,
    rel_remaining2_idx,
    rel_val_flag,
    rel_remaining2_flag
) = train_test_split(
    rel_remaining_idx,
    rel_remaining_flag,
    test_size=(20 / 35),
    random_state=RELIABILITY_RANDOM_STATE,
    stratify=rel_remaining_flag
)

# 20% -> 10% calibration + 10% final test
(
    rel_cal_idx,
    rel_test_idx
) = train_test_split(
    rel_remaining2_idx,
    test_size=0.50,
    random_state=RELIABILITY_RANDOM_STATE,
    stratify=rel_remaining2_flag
)

reliability_partitions = {
    "Train": rel_train_idx,
    "Validation": rel_val_idx,
    "Calibration": rel_cal_idx,
    "Test": rel_test_idx
}

print("RELIABILITY FOUR-WAY SPLIT")
print("=" * 80)

for name, idx in reliability_partitions.items():

    print(
        f"{name:12s}: "
        f"{len(idx):3d} "
        f"({100*len(idx)/len(reliability_union_idx):5.1f}%)"
    )

all_sets = [
    set(x)
    for x in reliability_partitions.values()
]

for i in range(len(all_sets)):

    for j in range(i + 1, len(all_sets)):

        assert all_sets[i].isdisjoint(
            all_sets[j]
        )

print(
    "\nPolymer-level leakage check: PASSED"
)

RELIABILITY FOUR-WAY SPLIT
Train       : 532 ( 65.0%)
Validation  : 123 ( 15.0%)
Calibration :  82 ( 10.0%)
Test        :  82 ( 10.0%)

Polymer-level leakage check: PASSED


In [43]:
reliability_label_rows = []

for partition_name, idx in reliability_partitions.items():

    row = {
        "Partition": partition_name,
        "Polymers": len(idx)
    }

    for target in core_targets:

        row[target] = int(
            metadata.loc[
                idx,
                target
            ]
            .notna()
            .sum()
        )

    row["Complete_triplets"] = int(
        metadata.loc[
            idx,
            core_targets
        ]
        .notna()
        .all(axis=1)
        .sum()
    )

    reliability_label_rows.append(
        row
    )

reliability_label_counts = pd.DataFrame(
    reliability_label_rows
)

print("RELIABILITY LABEL AVAILABILITY")
print("=" * 90)

display(
    reliability_label_counts
)

RELIABILITY LABEL AVAILABILITY


,Partition,Polymers,Tc,Density,Rg,Complete_triplets
0,Train,532,479,398,399,345
1,Validation,123,110,93,93,80
2,Calibration,82,78,57,56,53
3,Test,82,70,65,66,53


## 30. Reliability-Specific Feature Transformation

The reliability experiment uses a new data partition.

The dimensionality-reduction and feature-scaling transformations from Part A are therefore not reused.

A new 128-component TruncatedSVD transformation is fitted exclusively to Morgan features from the reliability-training partition.

The 36 interpretable descriptors are concatenated with these 128 latent Morgan components.

A StandardScaler is then fitted only on the reliability-training representation.

This preserves the same 164-dimensional architecture while preventing information leakage from validation, calibration, or test polymers.

In [44]:
# ============================================================
# Reliability-specific feature transformation
# ============================================================

reliability_svd = TruncatedSVD(
    n_components=N_MORGAN_COMPONENTS,
    random_state=RELIABILITY_RANDOM_STATE
)

reliability_svd.fit(
    X_morgan[
        rel_train_idx
    ]
)

X_rel_morgan_latent = (
    reliability_svd.transform(
        X_morgan
    )
)

X_rel_raw = np.hstack([
    X_interpretable,
    X_rel_morgan_latent
])

reliability_scaler = StandardScaler()

reliability_scaler.fit(
    X_rel_raw[
        rel_train_idx
    ]
)

X_reliability = (
    reliability_scaler
    .transform(
        X_rel_raw
    )
    .astype(np.float32)
)

print("RELIABILITY REPRESENTATION")
print("=" * 80)

print(
    f"Input dimensions:       "
    f"{X_reliability.shape[1]}"
)

print(
    f"SVD variance captured:  "
    f"{reliability_svd.explained_variance_ratio_.sum():.4f}"
)

print(
    f"NaN values:             "
    f"{np.isnan(X_reliability).sum()}"
)

print(
    f"Infinite values:        "
    f"{np.isinf(X_reliability).sum()}"
)

RELIABILITY REPRESENTATION
Input dimensions:       164
SVD variance captured:  0.9771
NaN values:             0
Infinite values:        0


## 31. Reliability-Specific Target Standardization

Target transformations are recalculated using only labels available in the reliability-training partition.

For each property:

$$
z =
\frac{y-\mu_{\mathrm{train}}}
{\sigma_{\mathrm{train}}}
$$

Validation, calibration, and test observations use these same training-derived parameters.

This ensures that no target-distribution information from later partitions influences neural-network fitting.

In [45]:
reliability_target_statistics = {}

Y_rel_standardized = np.full(
    (
        len(metadata),
        len(core_targets)
    ),
    np.nan,
    dtype=np.float32
)

Y_rel_mask = np.zeros(
    (
        len(metadata),
        len(core_targets)
    ),
    dtype=np.float32
)

for j, target in enumerate(
    core_targets
):

    train_values = (
        metadata.loc[
            rel_train_idx,
            target
        ]
        .dropna()
        .to_numpy(dtype=float)
    )

    mean_value = train_values.mean()

    std_value = train_values.std(
        ddof=1
    )

    reliability_target_statistics[
        target
    ] = {
        "mean": mean_value,
        "std": std_value
    }

    available = (
        metadata[target]
        .notna()
        .to_numpy()
    )

    values = (
        metadata.loc[
            available,
            target
        ]
        .to_numpy(dtype=float)
    )

    Y_rel_standardized[
        available,
        j
    ] = (
        (
            values
            - mean_value
        )
        / std_value
    )

    Y_rel_mask[
        available,
        j
    ] = 1.0


reliability_target_stats_table = pd.DataFrame([
    {
        "Target":
            target,

        "Train_mean":
            reliability_target_statistics[
                target
            ]["mean"],

        "Train_std":
            reliability_target_statistics[
                target
            ]["std"]
    }

    for target in core_targets
])


print(
    "RELIABILITY TARGET STANDARDIZATION"
)
print("=" * 80)

display(
    reliability_target_stats_table
    .round(4)
)

RELIABILITY TARGET STANDARDIZATION


,Target,Train_mean,Train_std
0,Tc,0.2584,0.0918
1,Density,0.9799,0.1496
2,Rg,16.2948,4.6485


## 32. Reliability Multitask Datasets

The four reliability partitions are converted into masked multitask datasets.

The same task-balanced masked loss developed in Part A is retained.

No calibration or test observation participates in model fitting or early stopping.

In [46]:
# ============================================================
# Reliability multitask datasets
# ============================================================

rel_train_dataset = PolymerMultitaskDataset(
    rel_train_idx,
    X_reliability,
    Y_rel_standardized,
    Y_rel_mask
)

rel_val_dataset = PolymerMultitaskDataset(
    rel_val_idx,
    X_reliability,
    Y_rel_standardized,
    Y_rel_mask
)

rel_cal_dataset = PolymerMultitaskDataset(
    rel_cal_idx,
    X_reliability,
    Y_rel_standardized,
    Y_rel_mask
)

rel_test_dataset = PolymerMultitaskDataset(
    rel_test_idx,
    X_reliability,
    Y_rel_standardized,
    Y_rel_mask
)

print("RELIABILITY DATASETS")
print("=" * 75)

print(f"Train:       {len(rel_train_dataset)}")
print(f"Validation:  {len(rel_val_dataset)}")
print(f"Calibration: {len(rel_cal_dataset)}")
print(f"Test:        {len(rel_test_dataset)}")

RELIABILITY DATASETS
Train:       532
Validation:  123
Calibration: 82
Test:        82


In [47]:
def build_reliability_loaders(
    seed
):

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        rel_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    val_loader = DataLoader(
        rel_val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    return (
        train_loader,
        val_loader
    )

## 34. Five-Member Multitask Reliability Ensemble

Five independently initialized multitask neural networks are trained using the frozen architecture and optimization protocol from Part A.

Each member sees the same polymer-level training and validation partitions but differs in:

- initial neural-network weights;
- mini-batch ordering;
- stochastic optimization trajectory.

The ensemble therefore provides five independent estimates for each polymer.

For a polymer $i$, the ensemble prediction is:

$$
\bar{y}_i
=
\frac{1}{M}
\sum_{m=1}^{M}
\hat{y}_{im}
$$

where $M=5$.

Ensemble disagreement is quantified using the standard deviation across member predictions:

$$
s_i
=
\sqrt{
\frac{1}{M-1}
\sum_{m=1}^{M}
(\hat{y}_{im}-\bar{y}_i)^2
}
$$

Large disagreement is interpreted as a candidate uncertainty signal.

It is not assumed in advance that disagreement must correlate with actual prediction error; this hypothesis is tested explicitly.

In [48]:
def train_reliability_multitask_model(
    seed
):

    set_all_seeds(seed)

    (
        train_loader,
        val_loader
    ) = build_reliability_loaders(
        seed
    )

    model = SharedMultitaskNet(
        input_dim=X_reliability.shape[1]
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    best_val_loss = np.inf
    best_state = None
    patience_counter = 0

    history = []

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        model.train()

        train_losses = []

        for (
            X_batch,
            y_batch,
            mask_batch
        ) in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)

            optimizer.zero_grad()

            predictions = model(
                X_batch
            )

            loss = masked_multitask_mse(
                predictions,
                y_batch,
                mask_batch
            )

            loss.backward()

            optimizer.step()

            train_losses.append(
                loss.item()
            )

        train_loss = np.mean(
            train_losses
        )

        val_loss = evaluate_multitask_loss(
            model,
            val_loader
        )

        history.append({
            "Epoch": epoch,
            "Train_loss": train_loss,
            "Validation_loss": val_loss
        })

        if (
            val_loss
            <
            best_val_loss - 1e-6
        ):

            best_val_loss = val_loss

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1

        if (
            patience_counter
            >= PATIENCE
        ):
            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history)
    )

In [49]:
reliability_models = {}
reliability_histories = {}

for seed in NEURAL_SEEDS:

    print(
        f"Training reliability ensemble "
        f"member — seed {seed}..."
    )

    (
        model,
        history
    ) = train_reliability_multitask_model(
        seed
    )

    reliability_models[
        seed
    ] = model

    reliability_histories[
        seed
    ] = history

    print(
        f"  epochs = {len(history)}, "
        f"best epoch = "
        f"{int(history.loc[history['Validation_loss'].idxmin(), 'Epoch'])}"
    )

print(
    "\nFive-member reliability ensemble complete."
)

Training reliability ensemble member — seed 42...
  epochs = 102, best epoch = 62
Training reliability ensemble member — seed 52...
  epochs = 195, best epoch = 155
Training reliability ensemble member — seed 62...
  epochs = 61, best epoch = 21
Training reliability ensemble member — seed 72...
  epochs = 122, best epoch = 82
Training reliability ensemble member — seed 82...
  epochs = 80, best epoch = 40

Five-member reliability ensemble complete.


In [50]:
reliability_training_rows = []

for seed, history in (
    reliability_histories.items()
):

    best_row = history.loc[
        history[
            "Validation_loss"
        ].idxmin()
    ]

    reliability_training_rows.append({
        "Seed":
            seed,

        "Epochs_run":
            len(history),

        "Best_epoch":
            int(
                best_row["Epoch"]
            ),

        "Best_validation_loss":
            best_row[
                "Validation_loss"
            ]
    })


reliability_training_stability = pd.DataFrame(
    reliability_training_rows
)

print(
    "RELIABILITY ENSEMBLE TRAINING AUDIT"
)
print("=" * 85)

display(
    reliability_training_stability
    .round(4)
)

RELIABILITY ENSEMBLE TRAINING AUDIT


,Seed,Epochs_run,Best_epoch,Best_validation_loss
0,42,102,62,0.2027
1,52,195,155,0.2055
2,62,61,21,0.2100
3,72,122,82,0.2078
4,82,80,40,0.2262


## 36. Ensemble Predictions

All five trained ensemble members generate predictions for the independent calibration and reliability-test partitions.

Predictions are transformed back to their original physical property scales before uncertainty and error analysis.

For each polymer and target, the following quantities are retained:

- observed property value;
- ensemble-mean prediction;
- ensemble prediction standard deviation;
- residual;
- absolute prediction error.

Only polymers with an experimental measurement for a given target are included in that target's reliability analysis.

In [51]:
def ensemble_predict_original_scale(
    indices
):

    member_predictions = []

    X_tensor = torch.tensor(
        X_reliability[
            indices
        ],
        dtype=torch.float32
    ).to(device)

    for seed in NEURAL_SEEDS:

        model = reliability_models[
            seed
        ]

        model.eval()

        with torch.no_grad():

            pred_std = (
                model(
                    X_tensor
                )
                .cpu()
                .numpy()
            )

        pred_original = np.zeros_like(
            pred_std,
            dtype=float
        )

        for j, target in enumerate(
            core_targets
        ):

            mean_value = (
                reliability_target_statistics[
                    target
                ]["mean"]
            )

            std_value = (
                reliability_target_statistics[
                    target
                ]["std"]
            )

            pred_original[
                :,
                j
            ] = (
                pred_std[
                    :,
                    j
                ]
                * std_value
                + mean_value
            )

        member_predictions.append(
            pred_original
        )

    return np.stack(
        member_predictions,
        axis=0
    )

In [52]:
calibration_member_predictions = (
    ensemble_predict_original_scale(
        rel_cal_idx
    )
)

test_member_predictions = (
    ensemble_predict_original_scale(
        rel_test_idx
    )
)


print(
    "Calibration prediction tensor:",
    calibration_member_predictions.shape
)

print(
    "Test prediction tensor:       ",
    test_member_predictions.shape
)

Calibration prediction tensor: (5, 82, 3)
Test prediction tensor:        (5, 82, 3)


In [53]:
calibration_ensemble_mean = (
    calibration_member_predictions
    .mean(axis=0)
)

calibration_ensemble_std = (
    calibration_member_predictions
    .std(
        axis=0,
        ddof=1
    )
)

test_ensemble_mean = (
    test_member_predictions
    .mean(axis=0)
)

test_ensemble_std = (
    test_member_predictions
    .std(
        axis=0,
        ddof=1
    )
)

In [54]:
reliability_tables = {}

for split_name, indices, pred_mean, pred_std in [

    (
        "Calibration",
        rel_cal_idx,
        calibration_ensemble_mean,
        calibration_ensemble_std
    ),

    (
        "Test",
        rel_test_idx,
        test_ensemble_mean,
        test_ensemble_std
    )
]:

    for j, target in enumerate(
        core_targets
    ):

        available = (
            metadata.loc[
                indices,
                target
            ]
            .notna()
            .to_numpy()
        )

        target_idx = (
            indices[
                available
            ]
        )

        y_true = (
            metadata.loc[
                target_idx,
                target
            ]
            .to_numpy(dtype=float)
        )

        y_pred = (
            pred_mean[
                available,
                j
            ]
        )

        uncertainty = (
            pred_std[
                available,
                j
            ]
        )

        residual = (
            y_true
            - y_pred
        )

        table = pd.DataFrame({
            "row_index":
                target_idx,

            "id":
                metadata.loc[
                    target_idx,
                    "id"
                ].to_numpy(),

            "Observed":
                y_true,

            "Ensemble_Mean":
                y_pred,

            "Ensemble_Std":
                uncertainty,

            "Residual":
                residual,

            "Absolute_Error":
                np.abs(
                    residual
                )
        })

        reliability_tables[
            (
                split_name,
                target
            )
        ] = table

In [55]:
print("RELIABILITY PREDICTION TABLES")
print("=" * 80)

for (
    split_name,
    target
), table in reliability_tables.items():

    print(
        f"{split_name:11s} | "
        f"{target:8s} | "
        f"n={len(table)}"
    )

RELIABILITY PREDICTION TABLES
Calibration | Tc       | n=78
Calibration | Density  | n=57
Calibration | Rg       | n=56
Test        | Tc       | n=70
Test        | Density  | n=65
Test        | Rg       | n=66


In [56]:
ensemble_point_rows = []

for (
    split_name,
    target
), table in reliability_tables.items():

    r2 = r2_score(
        table["Observed"],
        table["Ensemble_Mean"]
    )

    mae = mean_absolute_error(
        table["Observed"],
        table["Ensemble_Mean"]
    )

    rmse = np.sqrt(
        mean_squared_error(
            table["Observed"],
            table["Ensemble_Mean"]
        )
    )

    ensemble_point_rows.append({
        "Split":
            split_name,

        "Target":
            target,

        "n":
            len(table),

        "R2":
            r2,

        "MAE":
            mae,

        "RMSE":
            rmse,

        "Mean_Ensemble_Std":
            table[
                "Ensemble_Std"
            ].mean()
    })


ensemble_point_performance = pd.DataFrame(
    ensemble_point_rows
)

print(
    "RELIABILITY ENSEMBLE POINT PERFORMANCE"
)
print("=" * 100)

display(
    ensemble_point_performance
    .round(4)
)

RELIABILITY ENSEMBLE POINT PERFORMANCE


,Split,Target,n,R2,MAE,RMSE,Mean_Ensemble_Std
0,Calibration,Tc,78,0.6927,0.0295,0.0473,0.0075
1,Calibration,Density,57,0.9170,0.0209,0.0304,0.0110
2,Calibration,Rg,56,0.5979,1.6837,2.6131,0.3842
3,Test,Tc,70,0.7598,0.0263,0.0425,0.0081
4,Test,Density,65,0.9422,0.0258,0.0377,0.0121
5,Test,Rg,66,0.7188,1.7780,2.5337,0.4488


## 39. Ensemble Disagreement as a Reliability Signal

The ensemble standard deviation is useful only if larger model disagreement corresponds to larger realized prediction error.

Spearman rank correlation is used because the relationship between disagreement and error need not be linear.

For each target:

$$
\rho_s
=
\rho(
s_i,
|y_i-\bar{y}_i|
)
$$

A positive correlation indicates that predictions with greater ensemble disagreement tend to be less accurate.

The association is evaluated separately on:

- the calibration partition;
- the independent reliability-test partition.

The test-partition relationship provides the final out-of-sample assessment of ensemble disagreement as an uncertainty signal.

In [57]:
uncertainty_error_rows = []

for (
    split_name,
    target
), table in reliability_tables.items():

    rho, p_value = spearmanr(
        table[
            "Ensemble_Std"
        ],
        table[
            "Absolute_Error"
        ]
    )

    uncertainty_error_rows.append({
        "Split":
            split_name,

        "Target":
            target,

        "n":
            len(table),

        "Spearman_uncertainty_error":
            rho,

        "P_value":
            p_value,

        "Mean_uncertainty":
            table[
                "Ensemble_Std"
            ].mean(),

        "Median_uncertainty":
            table[
                "Ensemble_Std"
            ].median()
    })


uncertainty_error_summary = pd.DataFrame(
    uncertainty_error_rows
)

print(
    "ENSEMBLE DISAGREEMENT VS ABSOLUTE ERROR"
)
print("=" * 105)

display(
    uncertainty_error_summary
    .round(4)
)

ENSEMBLE DISAGREEMENT VS ABSOLUTE ERROR


,Split,Target,n,Spearman_uncertainty_error,P_value,Mean_uncertainty,Median_uncertainty
0,Calibration,Tc,78,0.3779,0.0006,0.0075,0.0063
1,Calibration,Density,57,0.5123,0.0000,0.0110,0.0084
2,Calibration,Rg,56,0.2801,0.0365,0.3842,0.3458
3,Test,Tc,70,0.3235,0.0063,0.0081,0.0068
4,Test,Density,65,0.3496,0.0043,0.0121,0.0100
5,Test,Rg,66,0.2244,0.0701,0.4488,0.4416


## 40. Absolute-Residual Split Conformal Prediction

Ensemble disagreement measures model-to-model variability, but it does not by itself guarantee calibrated uncertainty intervals.

Split conformal prediction is therefore used to construct prediction intervals with a target marginal coverage of 90%.

For each target, the calibration nonconformity score is:

$$
s_i =
|y_i-\hat{y}_i|
$$

where $\hat{y}_i$ is the ensemble-mean prediction.

For a desired miscoverage level $\alpha=0.10$, the conformal threshold is obtained from the finite-sample corrected calibration quantile.

The resulting interval for a new polymer is:

$$
[
\hat{y}-q_{0.90},
\hat{y}+q_{0.90}
]
$$

where $q_{0.90}$ is calculated exclusively from the independent calibration partition.

The reliability-test labels play no role in determining interval width.

Under the standard exchangeability assumption, split conformal prediction provides marginal finite-sample coverage rather than a guarantee for every individual polymer or every chemical subdomain.

In [58]:
# ============================================================
# Finite-sample conformal quantile
# ============================================================

CONFORMAL_ALPHA = 0.10
TARGET_COVERAGE = 1 - CONFORMAL_ALPHA


def conformal_quantile(
    scores,
    alpha=0.10
):

    scores = np.asarray(
        scores,
        dtype=float
    )

    n = len(scores)

    quantile_level = (
        np.ceil(
            (n + 1)
            * (1 - alpha)
        )
        / n
    )

    quantile_level = min(
        quantile_level,
        1.0
    )

    q = np.quantile(
        scores,
        quantile_level,
        method="higher"
    )

    return (
        q,
        quantile_level
    )


print(
    f"Target coverage: "
    f"{TARGET_COVERAGE:.0%}"
)

Target coverage: 90%


In [59]:
absolute_conformal_rows = []

absolute_conformal_thresholds = {}


for target in core_targets:

    calibration_table = (
        reliability_tables[
            ("Calibration", target)
        ]
    )

    calibration_scores = (
        calibration_table[
            "Absolute_Error"
        ]
        .to_numpy()
    )

    q, quantile_level = (
        conformal_quantile(
            calibration_scores,
            alpha=CONFORMAL_ALPHA
        )
    )

    absolute_conformal_thresholds[
        target
    ] = q

    absolute_conformal_rows.append({
        "Target":
            target,

        "Calibration_n":
            len(
                calibration_scores
            ),

        "Quantile_level":
            quantile_level,

        "Conformal_half_width":
            q,

        "Full_interval_width":
            2 * q
    })


absolute_conformal_calibration = (
    pd.DataFrame(
        absolute_conformal_rows
    )
)


print(
    "ABSOLUTE-RESIDUAL CONFORMAL CALIBRATION"
)
print("=" * 105)

display(
    absolute_conformal_calibration
    .round(4)
)

ABSOLUTE-RESIDUAL CONFORMAL CALIBRATION


,Target,Calibration_n,Quantile_level,Conformal_half_width,Full_interval_width
0,Tc,78,0.9231,0.0678,0.1356
1,Density,57,0.9298,0.0637,0.1275
2,Rg,56,0.9286,6.2292,12.4583


In [60]:
absolute_conformal_test_rows = []

absolute_conformal_test_tables = {}


for target in core_targets:

    table = (
        reliability_tables[
            ("Test", target)
        ]
        .copy()
    )

    q = (
        absolute_conformal_thresholds[
            target
        ]
    )

    table[
        "Conformal_Lower"
    ] = (
        table["Ensemble_Mean"]
        - q
    )

    table[
        "Conformal_Upper"
    ] = (
        table["Ensemble_Mean"]
        + q
    )

    table[
        "Covered"
    ] = (
        (
            table["Observed"]
            >=
            table["Conformal_Lower"]
        )
        &
        (
            table["Observed"]
            <=
            table["Conformal_Upper"]
        )
    )

    table[
        "Interval_Width"
    ] = (
        table[
            "Conformal_Upper"
        ]
        -
        table[
            "Conformal_Lower"
        ]
    )

    absolute_conformal_test_tables[
        target
    ] = table

    test_std = (
        table["Observed"]
        .std(ddof=1)
    )

    absolute_conformal_test_rows.append({
        "Target":
            target,

        "Test_n":
            len(table),

        "Target_coverage":
            TARGET_COVERAGE,

        "Empirical_coverage":
            table[
                "Covered"
            ].mean(),

        "Mean_interval_width":
            table[
                "Interval_Width"
            ].mean(),

        "Median_interval_width":
            table[
                "Interval_Width"
            ].median(),

        "Normalized_mean_width":
            (
                table[
                    "Interval_Width"
                ].mean()
                / test_std
            )
    })


absolute_conformal_test_summary = (
    pd.DataFrame(
        absolute_conformal_test_rows
    )
)


print(
    "90% ABSOLUTE-RESIDUAL CONFORMAL TEST PERFORMANCE"
)
print("=" * 110)

display(
    absolute_conformal_test_summary
    .round(4)
)

90% ABSOLUTE-RESIDUAL CONFORMAL TEST PERFORMANCE


,Target,Test_n,Target_coverage,Empirical_coverage,Mean_interval_width,Median_interval_width,Normalized_mean_width
0,Tc,70,0.9,0.9571,0.1356,0.1356,1.5521
1,Density,65,0.9,0.9231,0.1275,0.1275,0.8067
2,Rg,66,0.9,0.9848,12.4583,12.4583,2.5876


## 42. Disagreement-Scaled Conformal Prediction

The fixed-width conformal interval is calibrated but does not adapt to individual prediction difficulty.

Because ensemble disagreement was associated with realized error — particularly for Tc and Density — an adaptive conformal score is also evaluated:

$$
s_i
=
\frac{
|y_i-\hat{y}_i|
}{
\sigma_{\mathrm{ens},i}+\epsilon
}
$$

where $\sigma_{\mathrm{ens},i}$ is the standard deviation across the five neural-network predictions and $\epsilon$ is a small numerical constant preventing division by zero.

After calibration, the test interval becomes:

$$
[
\hat{y}_i
-
q(\sigma_{\mathrm{ens},i}+\epsilon),
\;
\hat{y}_i
+
q(\sigma_{\mathrm{ens},i}+\epsilon)
]
$$

This approach tests whether ensemble disagreement can allocate wider intervals to difficult predictions and narrower intervals to more stable predictions.

The method is evaluated for all three targets, but stronger adaptive behaviour is expected for Tc and Density because their disagreement–error associations reproduced on the independent reliability-test set.

In [61]:
# ============================================================
# Disagreement-scaled conformal calibration
# ============================================================

UNCERTAINTY_EPSILON = 1e-8

scaled_conformal_thresholds = {}
scaled_conformal_calibration_rows = []


for target in core_targets:

    calibration_table = (
        reliability_tables[
            ("Calibration", target)
        ]
    )

    scale = (
        calibration_table[
            "Ensemble_Std"
        ]
        .to_numpy()
        +
        UNCERTAINTY_EPSILON
    )

    scaled_scores = (
        calibration_table[
            "Absolute_Error"
        ]
        .to_numpy()
        / scale
    )

    q, quantile_level = (
        conformal_quantile(
            scaled_scores,
            alpha=CONFORMAL_ALPHA
        )
    )

    scaled_conformal_thresholds[
        target
    ] = q

    scaled_conformal_calibration_rows.append({
        "Target":
            target,

        "Calibration_n":
            len(
                calibration_table
            ),

        "Quantile_level":
            quantile_level,

        "Scaled_conformal_q":
            q,

        "Median_ensemble_std":
            calibration_table[
                "Ensemble_Std"
            ].median()
    })


scaled_conformal_calibration = (
    pd.DataFrame(
        scaled_conformal_calibration_rows
    )
)


print(
    "DISAGREEMENT-SCALED CONFORMAL CALIBRATION"
)
print("=" * 105)

display(
    scaled_conformal_calibration
    .round(4)
)

DISAGREEMENT-SCALED CONFORMAL CALIBRATION


,Target,Calibration_n,Quantile_level,Scaled_conformal_q,Median_ensemble_std
0,Tc,78,0.9231,15.3779,0.0063
1,Density,57,0.9298,5.7097,0.0084
2,Rg,56,0.9286,14.5211,0.3458


In [62]:
scaled_conformal_test_rows = []

scaled_conformal_test_tables = {}


for target in core_targets:

    table = (
        reliability_tables[
            ("Test", target)
        ]
        .copy()
    )

    q = (
        scaled_conformal_thresholds[
            target
        ]
    )

    local_scale = (
        table[
            "Ensemble_Std"
        ]
        +
        UNCERTAINTY_EPSILON
    )

    half_width = (
        q
        * local_scale
    )

    table[
        "Scaled_Lower"
    ] = (
        table[
            "Ensemble_Mean"
        ]
        -
        half_width
    )

    table[
        "Scaled_Upper"
    ] = (
        table[
            "Ensemble_Mean"
        ]
        +
        half_width
    )

    table[
        "Scaled_Covered"
    ] = (
        (
            table["Observed"]
            >=
            table["Scaled_Lower"]
        )
        &
        (
            table["Observed"]
            <=
            table["Scaled_Upper"]
        )
    )

    table[
        "Scaled_Interval_Width"
    ] = (
        2
        * half_width
    )

    scaled_conformal_test_tables[
        target
    ] = table

    test_std = (
        table[
            "Observed"
        ]
        .std(ddof=1)
    )

    scaled_conformal_test_rows.append({
        "Target":
            target,

        "Test_n":
            len(table),

        "Target_coverage":
            TARGET_COVERAGE,

        "Empirical_coverage":
            table[
                "Scaled_Covered"
            ].mean(),

        "Mean_interval_width":
            table[
                "Scaled_Interval_Width"
            ].mean(),

        "Median_interval_width":
            table[
                "Scaled_Interval_Width"
            ].median(),

        "Normalized_mean_width":
            (
                table[
                    "Scaled_Interval_Width"
                ].mean()
                / test_std
            )
    })


scaled_conformal_test_summary = (
    pd.DataFrame(
        scaled_conformal_test_rows
    )
)


print(
    "90% DISAGREEMENT-SCALED "
    "CONFORMAL TEST PERFORMANCE"
)
print("=" * 110)

display(
    scaled_conformal_test_summary
    .round(4)
)

90% DISAGREEMENT-SCALED CONFORMAL TEST PERFORMANCE


,Target,Test_n,Target_coverage,Empirical_coverage,Mean_interval_width,Median_interval_width,Normalized_mean_width
0,Tc,70,0.9,0.9857,0.2496,0.2083,2.8575
1,Density,65,0.9,0.9231,0.1380,0.1142,0.8731
2,Rg,66,0.9,0.9697,13.0332,12.8240,2.7070


In [63]:
conformal_comparison = (
    absolute_conformal_test_summary[
        [
            "Target",
            "Empirical_coverage",
            "Mean_interval_width",
            "Normalized_mean_width"
        ]
    ]
    .rename(
        columns={
            "Empirical_coverage":
                "Fixed_Coverage",

            "Mean_interval_width":
                "Fixed_Mean_Width",

            "Normalized_mean_width":
                "Fixed_Normalized_Width"
        }
    )
    .merge(
        scaled_conformal_test_summary[
            [
                "Target",
                "Empirical_coverage",
                "Mean_interval_width",
                "Normalized_mean_width"
            ]
        ]
        .rename(
            columns={
                "Empirical_coverage":
                    "Adaptive_Coverage",

                "Mean_interval_width":
                    "Adaptive_Mean_Width",

                "Normalized_mean_width":
                    "Adaptive_Normalized_Width"
            }
        ),
        on="Target"
    )
)


conformal_comparison[
    "Adaptive_width_change_pct"
] = (
    100
    * (
        conformal_comparison[
            "Adaptive_Mean_Width"
        ]
        -
        conformal_comparison[
            "Fixed_Mean_Width"
        ]
    )
    /
    conformal_comparison[
        "Fixed_Mean_Width"
    ]
)


print(
    "FIXED VS ADAPTIVE CONFORMAL INTERVALS"
)
print("=" * 120)

display(
    conformal_comparison.round(4)
)

FIXED VS ADAPTIVE CONFORMAL INTERVALS


,Target,Fixed_Coverage,Fixed_Mean_Width,Fixed_Normalized_Width,Adaptive_Coverage,Adaptive_Mean_Width,Adaptive_Normalized_Width,Adaptive_width_change_pct
0,Tc,0.9571,0.1356,1.5521,0.9857,0.2496,2.8575,84.1009
1,Density,0.9231,0.1275,0.8067,0.9231,0.1380,0.8731,8.2280
2,Rg,0.9848,12.4583,2.5876,0.9697,13.0332,2.7070,4.6142


In [64]:
adaptive_width_error_rows = []


for target in core_targets:

    table = (
        scaled_conformal_test_tables[
            target
        ]
    )

    rho, p_value = spearmanr(
        table[
            "Scaled_Interval_Width"
        ],
        table[
            "Absolute_Error"
        ]
    )

    adaptive_width_error_rows.append({
        "Target":
            target,

        "n":
            len(table),

        "Spearman_width_vs_error":
            rho,

        "P_value":
            p_value
    })


adaptive_width_error = pd.DataFrame(
    adaptive_width_error_rows
)


print(
    "ADAPTIVE INTERVAL WIDTH VS ACTUAL ERROR"
)
print("=" * 95)

display(
    adaptive_width_error.round(4)
)

ADAPTIVE INTERVAL WIDTH VS ACTUAL ERROR


,Target,n,Spearman_width_vs_error,P_value
0,Tc,70,0.3235,0.0063
1,Density,65,0.3496,0.0043
2,Rg,66,0.2244,0.0701


## 46. Chemical Support and Prediction-Level Uncertainty

Notebook 04 demonstrated that chemical novelty can reduce predictive reliability, but the strength of this effect depends strongly on the target property.

Notebook 05 has now produced a second reliability signal:

- ensemble disagreement.

The final reliability analysis asks whether these two signals describe the same phenomenon.

For each reliability-test polymer, chemical support is measured as the maximum binary Morgan Tanimoto similarity to any polymer in the reliability-training partition:

$$
S_{\max}(i)
=
\max_{j \in \mathcal{T}}
T(i,j)
$$

This is compared with:

- ensemble standard deviation;
- absolute prediction error;
- calibrated interval behaviour.

A negative relationship between chemical similarity and ensemble disagreement would indicate that neural uncertainty tends to increase in structurally unsupported regions.

However, the two measures need not be equivalent.

Chemical similarity measures structural support relative to known training chemistry, whereas ensemble disagreement measures instability among independently trained predictive models.

In [65]:
# ============================================================
# Binary Morgan fingerprints for reliability-support analysis
# ============================================================

X_morgan_binary = (
    X_morgan > 0
).astype(np.uint8)

rel_train_binary = (
    X_morgan_binary[
        rel_train_idx
    ]
)

rel_test_binary = (
    X_morgan_binary[
        rel_test_idx
    ]
)

print("RELIABILITY CHEMICAL-SUPPORT MATRICES")
print("=" * 80)

print(
    "Training fingerprints:",
    rel_train_binary.shape
)

print(
    "Test fingerprints:    ",
    rel_test_binary.shape
)

RELIABILITY CHEMICAL-SUPPORT MATRICES
Training fingerprints: (532, 2048)
Test fingerprints:     (82, 2048)


In [66]:
# ============================================================
# Maximum test-to-training Tanimoto similarity
# ============================================================

test_intersections = (
    rel_test_binary.astype(np.float32)
    @
    rel_train_binary.astype(np.float32).T
)

test_bit_counts = (
    rel_test_binary.sum(
        axis=1
    )
)

train_bit_counts = (
    rel_train_binary.sum(
        axis=1
    )
)

test_unions = (
    test_bit_counts[:, None]
    +
    train_bit_counts[None, :]
    -
    test_intersections
)

tanimoto_matrix = np.divide(
    test_intersections,
    test_unions,
    out=np.zeros_like(
        test_intersections,
        dtype=np.float32
    ),
    where=test_unions != 0
)

reliability_max_similarity = (
    tanimoto_matrix.max(
        axis=1
    )
)

reliability_top5_similarity = np.mean(
    np.partition(
        tanimoto_matrix,
        -5,
        axis=1
    )[:, -5:],
    axis=1
)


reliability_support_table = pd.DataFrame({
    "row_index":
        rel_test_idx,

    "Max_Train_Similarity":
        reliability_max_similarity,

    "Top5_Mean_Similarity":
        reliability_top5_similarity
})


print("RELIABILITY-TEST CHEMICAL SUPPORT")
print("=" * 90)

display(
    reliability_support_table[
        [
            "Max_Train_Similarity",
            "Top5_Mean_Similarity"
        ]
    ]
    .describe()
    .round(4)
)

RELIABILITY-TEST CHEMICAL SUPPORT


,Max_Train_Similarity,Top5_Mean_Similarity
count,82.0000,82.0000
mean,0.5843,0.4759
std,0.2313,0.2077
min,0.2083,0.1561
25%,0.3947,0.3320
50%,0.5179,0.4277
75%,0.7644,0.5503
max,1.0000,1.0000


In [67]:
reliability_chemistry_tables = {}

for target in core_targets:

    prediction_table = (
        reliability_tables[
            ("Test", target)
        ]
        .copy()
    )

    combined = prediction_table.merge(
        reliability_support_table,
        on="row_index",
        how="left",
        validate="one_to_one"
    )

    assert (
        combined[
            "Max_Train_Similarity"
        ]
        .notna()
        .all()
    )

    reliability_chemistry_tables[
        target
    ] = combined


print("CHEMISTRY–RELIABILITY TABLES")
print("=" * 80)

for target, table in (
    reliability_chemistry_tables.items()
):

    print(
        f"{target:8s}: "
        f"{len(table)} polymers"
    )

CHEMISTRY–RELIABILITY TABLES
Tc      : 70 polymers
Density : 65 polymers
Rg      : 66 polymers


## 48. Chemical Novelty Versus Ensemble Uncertainty

Chemical support and ensemble disagreement are now compared directly.

Three associations are evaluated for each property:

$$
S_{\max}
\leftrightarrow
\sigma_{\mathrm{ensemble}}
$$

$$
S_{\max}
\leftrightarrow
|y-\hat{y}|
$$

and

$$
\sigma_{\mathrm{ensemble}}
\leftrightarrow
|y-\hat{y}|
$$

The first tests whether structurally unsupported polymers receive larger model disagreement.

The second reproduces the chemistry-aware reliability question from Notebook 04 within the independent Notebook 05 reliability experiment.

The third tests whether prediction-level uncertainty identifies actual difficult predictions.

These relationships are not assumed to be identical.

In [68]:
chemistry_uncertainty_rows = []

for target, table in (
    reliability_chemistry_tables.items()
):

    rho_sim_unc, p_sim_unc = spearmanr(
        table[
            "Max_Train_Similarity"
        ],
        table[
            "Ensemble_Std"
        ]
    )

    rho_sim_err, p_sim_err = spearmanr(
        table[
            "Max_Train_Similarity"
        ],
        table[
            "Absolute_Error"
        ]
    )

    rho_unc_err, p_unc_err = spearmanr(
        table[
            "Ensemble_Std"
        ],
        table[
            "Absolute_Error"
        ]
    )

    chemistry_uncertainty_rows.append({
        "Target":
            target,

        "n":
            len(table),

        "Similarity_vs_Uncertainty_rho":
            rho_sim_unc,

        "Similarity_vs_Uncertainty_p":
            p_sim_unc,

        "Similarity_vs_Error_rho":
            rho_sim_err,

        "Similarity_vs_Error_p":
            p_sim_err,

        "Uncertainty_vs_Error_rho":
            rho_unc_err,

        "Uncertainty_vs_Error_p":
            p_unc_err
    })


chemistry_uncertainty_summary = pd.DataFrame(
    chemistry_uncertainty_rows
)


print(
    "CHEMICAL SUPPORT, UNCERTAINTY, AND ERROR"
)
print("=" * 120)

display(
    chemistry_uncertainty_summary.round(4)
)

CHEMICAL SUPPORT, UNCERTAINTY, AND ERROR


,Target,n,Similarity_vs_Uncertainty_rho,Similarity_vs_Uncertainty_p,Similarity_vs_Error_rho,Similarity_vs_Error_p,Uncertainty_vs_Error_rho,Uncertainty_vs_Error_p
0,Tc,70,-0.0708,0.5600,0.002,0.9870,0.3235,0.0063
1,Density,65,-0.4545,0.0001,-0.413,0.0006,0.3496,0.0043
2,Rg,66,-0.0193,0.8779,-0.264,0.0322,0.2244,0.0701


## 49. Reliability in Low- and High-Support Chemistry

The reliability-test set is substantially smaller than the Notebook 04 holdout.

To avoid unstable subdivision into many small groups, chemical support is therefore summarized using two broad regimes:

- **Lower support:** maximum training similarity < 0.6
- **Higher support:** maximum training similarity >= 0.6

For each regime, the analysis compares:

- number of polymers;
- absolute prediction error;
- ensemble disagreement;
- fixed conformal coverage.

The purpose is to determine whether calibrated uncertainty remains reliable across chemically different support conditions.

In [69]:
support_regime_rows = []

for target in core_targets:

    base_table = (
        reliability_chemistry_tables[
            target
        ]
        .copy()
    )

    conformal_table = (
        absolute_conformal_test_tables[
            target
        ][
            [
                "row_index",
                "Covered",
                "Interval_Width"
            ]
        ]
    )

    table = base_table.merge(
        conformal_table,
        on="row_index",
        how="left",
        validate="one_to_one"
    )

    table["Support_Regime"] = np.where(
        table[
            "Max_Train_Similarity"
        ] < 0.6,
        "Lower support (<0.6)",
        "Higher support (>=0.6)"
    )

    for regime, group in (
        table.groupby(
            "Support_Regime"
        )
    ):

        support_regime_rows.append({
            "Target":
                target,

            "Support_Regime":
                regime,

            "n":
                len(group),

            "Mean_similarity":
                group[
                    "Max_Train_Similarity"
                ].mean(),

            "Mean_absolute_error":
                group[
                    "Absolute_Error"
                ].mean(),

            "Mean_ensemble_std":
                group[
                    "Ensemble_Std"
                ].mean(),

            "Conformal_coverage":
                group[
                    "Covered"
                ].mean(),

            "Mean_interval_width":
                group[
                    "Interval_Width"
                ].mean()
        })


support_regime_summary = pd.DataFrame(
    support_regime_rows
)


print(
    "RELIABILITY BY CHEMICAL-SUPPORT REGIME"
)
print("=" * 120)

display(
    support_regime_summary.round(4)
)

RELIABILITY BY CHEMICAL-SUPPORT REGIME


,Target,Support_Regime,n,Mean_similarity,Mean_absolute_error,Mean_ensemble_std,Conformal_coverage,Mean_interval_width
0,Tc,Higher support (>=0.6),25,0.8708,0.0336,0.0086,0.9200,0.1356
1,Tc,Lower support (<0.6),45,0.4358,0.0223,0.0078,0.9778,0.1356
2,Density,Higher support (>=0.6),25,0.8459,0.0166,0.0091,0.9600,0.1275
3,Density,Lower support (<0.6),40,0.4286,0.0316,0.0139,0.9000,0.1275
4,Rg,Higher support (>=0.6),25,0.8459,1.5845,0.4994,1.0000,12.4583
5,Rg,Lower support (<0.6),41,0.4320,1.8960,0.4179,0.9756,12.4583


## 50. Complementarity of Reliability Signals

Chemical support and ensemble disagreement measure different aspects of predictive reliability.

To summarize their practical behaviour, prediction error and ensemble disagreement in lower-support chemistry are compared with those in higher-support chemistry.

For each target:

$$
R_{\mathrm{error}}
=
\frac{
\mathrm{MAE}_{\mathrm{low\ support}}
}{
\mathrm{MAE}_{\mathrm{high\ support}}
}
$$

and

$$
R_{\mathrm{uncertainty}}
=
\frac{
\overline{\sigma}_{\mathrm{low\ support}}
}{
\overline{\sigma}_{\mathrm{high\ support}}
}
$$

Values above one indicate larger error or greater ensemble disagreement in chemically less-supported regions.

This analysis helps distinguish targets for which chemical novelty and neural uncertainty provide redundant information from targets for which they provide complementary information.

In [70]:
# ============================================================
# Reliability signal complementarity
# ============================================================

reliability_signal_rows = []

for target in core_targets:

    target_table = (
        support_regime_summary[
            support_regime_summary["Target"] == target
        ]
    )

    low = target_table[
        target_table["Support_Regime"]
        == "Lower support (<0.6)"
    ].iloc[0]

    high = target_table[
        target_table["Support_Regime"]
        == "Higher support (>=0.6)"
    ].iloc[0]

    reliability_signal_rows.append({
        "Target":
            target,

        "LowSupport_Error_Ratio":
            (
                low["Mean_absolute_error"]
                /
                high["Mean_absolute_error"]
            ),

        "LowSupport_Uncertainty_Ratio":
            (
                low["Mean_ensemble_std"]
                /
                high["Mean_ensemble_std"]
            ),

        "LowSupport_Coverage":
            low["Conformal_coverage"],

        "HighSupport_Coverage":
            high["Conformal_coverage"]
    })


reliability_signal_complementarity = (
    pd.DataFrame(
        reliability_signal_rows
    )
)


print("COMPLEMENTARITY OF RELIABILITY SIGNALS")
print("=" * 100)

display(
    reliability_signal_complementarity.round(4)
)

COMPLEMENTARITY OF RELIABILITY SIGNALS


,Target,LowSupport_Error_Ratio,LowSupport_Uncertainty_Ratio,LowSupport_Coverage,HighSupport_Coverage
0,Tc,0.6648,0.9120,0.9778,0.92
1,Density,1.9071,1.5310,0.9000,0.96
2,Rg,1.1966,0.8368,0.9756,1.00


## 51. Selective Prediction and Risk–Coverage Analysis

A useful uncertainty score should allow difficult predictions to be identified before their true experimental values are known.

This is evaluated using selective prediction.

For each target, reliability-test polymers are ranked from lowest to highest ensemble disagreement.

Increasing fractions of the most uncertain predictions are then excluded, and predictive error is recalculated on the retained subset.

The retained fraction is referred to as coverage in the selective-prediction sense:

$$
C =
\frac{
N_{\mathrm{retained}}
}{
N_{\mathrm{total}}
}
$$

This is distinct from conformal interval coverage.

If ensemble disagreement is informative, prediction error should generally decrease as only the most confident predictions are retained.

This analysis evaluates whether ensemble uncertainty has practical value as a triage or confidence-ranking tool rather than relying only on correlation coefficients.

In [71]:
# ============================================================
# Selective prediction / risk–coverage analysis
# ============================================================

retained_fractions = [
    1.00,
    0.80,
    0.60,
    0.40
]

selective_prediction_rows = []

for target in core_targets:

    table = (
        reliability_tables[
            ("Test", target)
        ]
        .copy()
        .sort_values(
            "Ensemble_Std",
            ascending=True
        )
        .reset_index(drop=True)
    )

    n_total = len(table)

    for fraction in retained_fractions:

        n_keep = max(
            1,
            int(
                np.ceil(
                    fraction * n_total
                )
            )
        )

        retained = table.iloc[
            :n_keep
        ]

        mae = mean_absolute_error(
            retained["Observed"],
            retained["Ensemble_Mean"]
        )

        rmse = np.sqrt(
            mean_squared_error(
                retained["Observed"],
                retained["Ensemble_Mean"]
            )
        )

        selective_prediction_rows.append({
            "Target":
                target,

            "Retained_fraction":
                fraction,

            "n_retained":
                n_keep,

            "MAE":
                mae,

            "RMSE":
                rmse,

            "Mean_ensemble_std":
                retained[
                    "Ensemble_Std"
                ].mean()
        })


selective_prediction = pd.DataFrame(
    selective_prediction_rows
)

print(
    "SELECTIVE PREDICTION / RISK–COVERAGE"
)
print("=" * 105)

display(
    selective_prediction.round(4)
)

SELECTIVE PREDICTION / RISK–COVERAGE


,Target,Retained_fraction,n_retained,MAE,RMSE,Mean_ensemble_std
0,Tc,1.0,70,0.0263,0.0425,0.0081
1,Tc,0.8,56,0.0213,0.0270,0.0064
2,Tc,0.6,42,0.0206,0.0269,0.0053
3,Tc,0.4,28,0.0159,0.0207,0.0046
4,Density,1.0,65,0.0258,0.0377,0.0121
5,Density,0.8,52,0.0217,0.0281,0.0090
6,Density,0.6,39,0.0194,0.0263,0.0072
7,Density,0.4,26,0.0173,0.0257,0.0058
8,Rg,1.0,66,1.7780,2.5337,0.4488
9,Rg,0.8,53,1.7927,2.6313,0.3630


In [72]:
selective_gain_rows = []

for target in core_targets:

    subset = selective_prediction[
        selective_prediction["Target"]
        == target
    ]

    baseline = subset[
        subset["Retained_fraction"]
        == 1.00
    ].iloc[0]

    for _, row in subset.iterrows():

        selective_gain_rows.append({
            "Target":
                target,

            "Retained_fraction":
                row[
                    "Retained_fraction"
                ],

            "MAE_reduction_pct":
                100
                * (
                    baseline["MAE"]
                    - row["MAE"]
                )
                / baseline["MAE"],

            "RMSE_reduction_pct":
                100
                * (
                    baseline["RMSE"]
                    - row["RMSE"]
                )
                / baseline["RMSE"]
        })


selective_prediction_gain = pd.DataFrame(
    selective_gain_rows
)

print(
    "ERROR REDUCTION AFTER REJECTING "
    "HIGH-UNCERTAINTY PREDICTIONS"
)
print("=" * 105)

display(
    selective_prediction_gain.round(2)
)

ERROR REDUCTION AFTER REJECTING HIGH-UNCERTAINTY PREDICTIONS


,Target,Retained_fraction,MAE_reduction_pct,RMSE_reduction_pct
0,Tc,1.0,0.00,0.00
1,Tc,0.8,19.26,36.52
2,Tc,0.6,21.90,36.80
3,Tc,0.4,39.84,51.29
4,Density,1.0,0.00,0.00
5,Density,0.8,16.00,25.45
6,Density,0.6,24.85,30.22
7,Density,0.4,32.99,31.82
8,Rg,1.0,0.00,0.00
9,Rg,0.8,-0.82,-3.85


## 52. Statistical Uncertainty in Conformal Coverage

Empirical conformal coverage is estimated from finite reliability-test samples.

Observed coverage percentages therefore have sampling uncertainty.

Wilson binomial confidence intervals are calculated for the fixed conformal coverage estimates.

These intervals do not alter the conformal procedure itself; they quantify uncertainty in the empirical coverage measured on the finite test set.

In [73]:
# ============================================================
# Wilson 95% confidence intervals for empirical coverage
# ============================================================

from statsmodels.stats.proportion import (
    proportion_confint
)

coverage_uncertainty_rows = []

for target in core_targets:

    table = (
        absolute_conformal_test_tables[
            target
        ]
    )

    n = len(table)

    n_covered = int(
        table["Covered"].sum()
    )

    coverage = (
        n_covered / n
    )

    ci_low, ci_high = (
        proportion_confint(
            count=n_covered,
            nobs=n,
            alpha=0.05,
            method="wilson"
        )
    )

    coverage_uncertainty_rows.append({
        "Target":
            target,

        "Test_n":
            n,

        "Covered_n":
            n_covered,

        "Empirical_coverage":
            coverage,

        "Coverage_95CI_low":
            ci_low,

        "Coverage_95CI_high":
            ci_high
    })


conformal_coverage_uncertainty = (
    pd.DataFrame(
        coverage_uncertainty_rows
    )
)


print(
    "FIXED CONFORMAL COVERAGE WITH "
    "95% CONFIDENCE INTERVALS"
)
print("=" * 105)

display(
    conformal_coverage_uncertainty
    .round(4)
)

FIXED CONFORMAL COVERAGE WITH 95% CONFIDENCE INTERVALS


,Target,Test_n,Covered_n,Empirical_coverage,Coverage_95CI_low,Coverage_95CI_high
0,Tc,70,67,0.9571,0.8814,0.9853
1,Density,65,60,0.9231,0.8322,0.9667
2,Rg,66,65,0.9848,0.9190,0.9973


## 53. Notebook 05 Conclusions

Notebook 05 investigated two complementary questions:

1. whether related polymer properties benefit from shared representation learning;
2. whether individual predictions can be accompanied by scientifically meaningful reliability information.

---

### Shared-learning structure

Tc, Density, and Rg formed the strongest shared measurement block in the dataset.

Among 819 polymers containing at least one of these properties:

- 531 contained complete Tc–Density–Rg measurements;
- 202 contained Tc only;
- 79 contained Density and Rg;
- and smaller numbers contained other partial combinations.

Restricting multitask learning to complete triplets would therefore discard 288 polymers containing useful experimental information.

The target values themselves showed heterogeneous relationships.

Tc was moderately negatively associated with Density and positively associated with Rg, while Density and Rg were only weakly related directly.

The three properties therefore provided a scientifically plausible but nontrivial test of shared representation learning.

---

### Controlled single-task versus multitask comparison

Matched single-task and multitask neural networks used the same:

- 164-dimensional compact hybrid representation;
- polymer-level data partitions;
- hidden-layer architecture;
- regularization;
- target standardization;
- optimizer;
- early-stopping procedure;
- and five random seeds.

Mean multitask R² was slightly higher than mean matched single-task R²:

- Tc: 0.7739 versus 0.7728;
- Density: 0.5425 versus 0.5298;
- Rg: 0.7264 versus 0.7177.

However, these average gains were not consistently reproduced across seeds.

Multitask learning produced higher R² than the matched single-task model in only two of five seeds for every target, and median seed-level changes in R² were slightly negative.

The results therefore do not support a claim of strong or universally positive transfer.

Instead, shared learning produced comparable performance with occasional improvements and no evidence of systematic negative-transfer collapse.

---

### Partial labels provide real value

Masked multitask learning clearly outperformed complete-case multitask learning.

Mean R² values were approximately:

- Density: 0.5425 masked versus 0.5075 complete-case;
- Rg: 0.7264 masked versus 0.6840 complete-case;
- Tc: 0.7739 masked versus 0.7562 complete-case.

Thus, the 288 partially labelled polymers contained useful predictive information.

The strongest practical advantage of multitask learning in this dataset is therefore its ability to exploit incomplete experimental records rather than a dramatic universal accuracy gain over single-task models.

---

### Neural training stability

All single-task, masked-multitask, and reliability-ensemble models converged normally under validation-based early stopping.

No network reached the 500-epoch maximum.

The transfer results therefore cannot be explained by failure to converge or inadequate optimization time.

---

### Independent predictive-reliability experiment

A separate polymer-level reliability design was constructed using:

- 65% training;
- 15% validation;
- 10% conformal calibration;
- 10% final reliability testing.

Feature transformation, target standardization, neural fitting, early stopping, calibration, and testing were strictly separated.

Five independently initialized multitask models formed the final predictive ensemble.

The ensemble-mean reliability-test performance was:

- Tc: R² ≈ 0.760;
- Density: R² ≈ 0.942;
- Rg: R² ≈ 0.719.

---

### Ensemble disagreement identifies difficult predictions for Tc and Density

Ensemble prediction standard deviation was significantly associated with realized absolute error on the independent reliability-test set for:

- Tc: Spearman rho ≈ +0.324;
- Density: Spearman rho ≈ +0.350.

Rg showed a weaker positive relationship:

- rho ≈ +0.224;
- p ≈ 0.07.

Thus, ensemble disagreement provides a meaningful prediction-level reliability ranking for Tc and Density but is substantially weaker for Rg.

---

### Selective prediction confirms practical usefulness

The uncertainty ranking was tested operationally by progressively rejecting the most uncertain predictions.

For Tc, retaining only the 40% lowest-disagreement predictions reduced:

- MAE by approximately 39.8%;
- RMSE by approximately 51.3%.

For Density, retaining the lowest-uncertainty 40% reduced:

- MAE by approximately 33.0%;
- RMSE by approximately 31.8%.

This demonstrates that ensemble disagreement can be used to identify subsets of Tc and Density predictions with substantially lower expected error.

Rg did not show a stable risk–coverage relationship.

Removing high-disagreement Rg predictions produced only small or inconsistent improvements, reinforcing the conclusion that neural ensemble disagreement is not a strong standalone reliability indicator for this property.

---

### Calibrated split-conformal intervals

Independent calibration data were used to construct nominal 90% fixed split-conformal prediction intervals.

Observed reliability-test coverage was:

- Tc: 95.7%;
- Density: 92.3%;
- Rg: 98.5%.

Wilson 95% confidence intervals for empirical coverage were approximately:

- Tc: 88.1–98.5%;
- Density: 83.2–96.7%;
- Rg: 91.9–99.7%.

The nominal 90% level lies within the finite-sample confidence intervals for Tc and Density.

Rg remained strongly conservative, with its entire empirical-coverage confidence interval above 90%.

These intervals should be interpreted as marginal population-level uncertainty guarantees rather than guarantees for every individual polymer or chemical subdomain.

---

### Adaptive conformal intervals did not improve efficiency

A disagreement-scaled conformal procedure was evaluated to determine whether ensemble disagreement could be used directly to assign adaptive interval widths.

Although adaptive width remained associated with realized error for Tc and Density, the intervals were wider on average than the fixed conformal intervals:

- Tc: approximately 84% wider;
- Density: approximately 8% wider;
- Rg: approximately 5% wider.

The adaptive method therefore did not provide a superior global coverage–efficiency trade-off.

Fixed absolute-residual split-conformal intervals are retained as the primary calibrated uncertainty method.

Ensemble disagreement is retained separately as a relative reliability-ranking signal.

---

### Chemical support and prediction-level uncertainty are complementary

Maximum training-set Morgan similarity was compared with ensemble disagreement and actual prediction error.

Density showed alignment among all three quantities:

- lower chemical support corresponded to larger prediction error;
- lower chemical support corresponded to larger neural disagreement;
- and larger neural disagreement corresponded to larger error.

Lower-support Density polymers had approximately 1.91 times the mean absolute error and 1.53 times the ensemble disagreement of higher-support polymers.

Tc behaved differently.

Chemical similarity was associated with neither prediction error nor ensemble disagreement, yet ensemble disagreement significantly predicted realized error.

For Tc, prediction-level uncertainty therefore identifies difficulty that nearest-neighbour fingerprint similarity does not capture.

Rg showed the opposite tendency.

Lower chemical support was associated with larger prediction error, but ensemble disagreement did not systematically increase in low-support chemistry and showed only a weak relationship with error.

Chemical-domain support is therefore currently a more useful reliability indicator for Rg than neural disagreement.

---

### Reliability is multidimensional

No single uncertainty quantity performed universally across the three properties.

The final reliability framework therefore preserves four complementary outputs:

1. **Ensemble-mean point prediction**
2. **90% fixed split-conformal prediction interval**
3. **Ensemble standard deviation as a relative prediction-level reliability score**
4. **Maximum training-set chemical similarity as a structural-domain support score**

These quantities answer different questions:

- What value does the model predict?
- What range is calibrated to contain the true value at the chosen marginal coverage level?
- Do independently trained models agree?
- Is the polymer structurally supported by the training chemistry?

---

### Principal scientific conclusion

**Shared learning is valuable primarily because it allows partially labelled polymer data to be used efficiently; evidence for a large universal accuracy advantage over matched single-task learning is weak.**

**Predictive reliability is property dependent and cannot be represented adequately by a single uncertainty measure.**

For Tc and Density, ensemble disagreement provides useful prediction-level risk ranking.

For Rg, chemical-domain support is comparatively more informative.

Fixed split-conformal prediction provides the calibrated uncertainty layer across all three properties.

A reliable polymer-informatics prediction should therefore report not only a predicted value, but also calibrated uncertainty, model disagreement, and chemical-domain support.

These outputs form the reliability layer for Notebook 06, where structural interpretation, property landscapes, chemical-domain coverage, uncertainty, and cross-property relationships are integrated into the final Polymer Informatics Atlas.

## 54. Export Notebook 05

The complete outputs from the shared-learning and predictive-reliability analyses are exported for reproducibility and downstream integration in Notebook 06.

The export preserves:

- target-overlap and property-relationship analyses;
- single-task, complete-case multitask, and masked-multitask comparisons;
- seed-level transfer results;
- neural training-stability diagnostics;
- reliability partitions and transformations;
- ensemble predictions and uncertainty analyses;
- fixed and adaptive conformal calibration results;
- chemical-support reliability analyses;
- selective-prediction results;
- empirical conformal-coverage uncertainty;
- exact polymer partitions;
- fitted feature transformations;
- and the five-member reliability ensemble.

These artifacts allow Notebook 06 to use the finalized results without repeating the computationally expensive Notebook 05 workflow.

In [74]:
# ============================================================
# Create clean Notebook 05 output directory
# ============================================================

from pathlib import Path
import shutil
import json
import joblib

notebook05_output_dir = Path(
    "polymer_atlas_notebook05_outputs"
)

if notebook05_output_dir.exists():

    shutil.rmtree(
        notebook05_output_dir
    )

notebook05_output_dir.mkdir(
    exist_ok=True
)

print(
    f"Clean output directory created: "
    f"{notebook05_output_dir}"
)

Clean output directory created: polymer_atlas_notebook05_outputs


In [75]:
# ============================================================
# Export Part A — shared-learning results
# ============================================================

part_a_exports = {

    "target_overlap_matrix.csv":
        overlap_matrix.reset_index(),

    "core_target_relationships.csv":
        core_relationships,

    "missing_label_patterns.csv":
        missing_pattern_table,

    "shared_learning_partition_counts.csv":
        partition_label_counts,

    "target_standardization_partA.csv":
        target_statistics_table,

    "neural_transfer_results_all_seeds.csv":
        transfer_results,

    "neural_transfer_summary.csv":
        transfer_summary,

    "transfer_effect_summary.csv":
        transfer_effect,

    "paired_seed_transfer.csv":
        paired_transfer,

    "paired_transfer_robustness_summary.csv":
        paired_transfer_summary,

    "training_stability.csv":
        training_stability,

    "training_stability_summary.csv":
        training_stability_summary,

    "complete_case_multitask_results.csv":
        complete_case_results,

    "three_way_learning_summary.csv":
        three_way_summary
}


for filename, dataframe in (
    part_a_exports.items()
):

    dataframe.to_csv(
        notebook05_output_dir
        / filename,
        index=False
    )


print(
    f"Part A tables exported: "
    f"{len(part_a_exports)}"
)

Part A tables exported: 14


In [76]:
# ============================================================
# Export Part B — reliability results
# ============================================================

part_b_exports = {

    "reliability_partition_counts.csv":
        reliability_label_counts,

    "reliability_target_standardization.csv":
        reliability_target_stats_table,

    "reliability_training_stability.csv":
        reliability_training_stability,

    "ensemble_point_performance.csv":
        ensemble_point_performance,

    "ensemble_uncertainty_error_correlations.csv":
        uncertainty_error_summary,

    "absolute_conformal_calibration.csv":
        absolute_conformal_calibration,

    "absolute_conformal_test_summary.csv":
        absolute_conformal_test_summary,

    "scaled_conformal_calibration.csv":
        scaled_conformal_calibration,

    "scaled_conformal_test_summary.csv":
        scaled_conformal_test_summary,

    "fixed_vs_adaptive_conformal.csv":
        conformal_comparison,

    "adaptive_width_error_correlations.csv":
        adaptive_width_error,

    "chemistry_uncertainty_summary.csv":
        chemistry_uncertainty_summary,

    "chemical_support_regime_summary.csv":
        support_regime_summary,

    "reliability_signal_complementarity.csv":
        reliability_signal_complementarity,

    "selective_prediction.csv":
        selective_prediction,

    "selective_prediction_error_reduction.csv":
        selective_prediction_gain,

    "conformal_coverage_uncertainty.csv":
        conformal_coverage_uncertainty
}


for filename, dataframe in (
    part_b_exports.items()
):

    dataframe.to_csv(
        notebook05_output_dir
        / filename,
        index=False
    )


print(
    f"Part B tables exported: "
    f"{len(part_b_exports)}"
)

Part B tables exported: 17


In [77]:
# ============================================================
# Export raw calibration and reliability-test predictions
# ============================================================

for target in core_targets:

    reliability_tables[
        ("Calibration", target)
    ].to_csv(
        notebook05_output_dir
        / f"{target}_calibration_predictions.csv",
        index=False
    )

    reliability_tables[
        ("Test", target)
    ].to_csv(
        notebook05_output_dir
        / f"{target}_reliability_test_predictions.csv",
        index=False
    )


print(
    "Calibration and reliability-test "
    "prediction tables exported."
)

Calibration and reliability-test prediction tables exported.


In [79]:
# ============================================================
# Export fixed-conformal reliability-test tables
# ============================================================

for target in core_targets:

    absolute_conformal_test_tables[
        target
    ].to_csv(
        notebook05_output_dir
        / f"{target}_fixed_conformal_test.csv",
        index=False
    )


print(
    "Fixed conformal test tables exported."
)

Fixed conformal test tables exported.


In [80]:
# ============================================================
# Export adaptive-conformal reliability-test tables
# ============================================================

for target in core_targets:

    scaled_conformal_test_tables[
        target
    ].to_csv(
        notebook05_output_dir
        / f"{target}_adaptive_conformal_test.csv",
        index=False
    )


print(
    "Adaptive conformal test tables exported."
)

Adaptive conformal test tables exported.


In [81]:
# ============================================================
# Export chemical-support + reliability tables
# ============================================================

for target in core_targets:

    reliability_chemistry_tables[
        target
    ].to_csv(
        notebook05_output_dir
        / f"{target}_chemistry_reliability_test.csv",
        index=False
    )


print(
    "Chemistry–reliability tables exported."
)

Chemistry–reliability tables exported.


In [82]:
# ============================================================
# Export exact Notebook 05 polymer partitions
# ============================================================

partition_rows = []

partition_sets = {

    "PartA_Train":
        train_idx,

    "PartA_Validation":
        val_idx,

    "PartA_Test":
        test_idx,

    "Reliability_Train":
        rel_train_idx,

    "Reliability_Validation":
        rel_val_idx,

    "Reliability_Calibration":
        rel_cal_idx,

    "Reliability_Test":
        rel_test_idx
}


for partition_name, idx in (
    partition_sets.items()
):

    for row_idx in idx:

        partition_rows.append({

            "Partition":
                partition_name,

            "row_index":
                int(row_idx),

            "id":
                metadata.loc[
                    row_idx,
                    "id"
                ]
        })


notebook05_partitions = pd.DataFrame(
    partition_rows
)


notebook05_partitions.to_csv(
    notebook05_output_dir
    / "notebook05_polymer_partitions.csv",
    index=False
)


print(
    "Exact Notebook 05 polymer "
    "partitions exported."
)

Exact Notebook 05 polymer partitions exported.


In [83]:
# ============================================================
# Export fitted feature transformations
# ============================================================

joblib.dump(
    morgan_svd,
    notebook05_output_dir
    / "partA_morgan_svd.joblib"
)

joblib.dump(
    feature_scaler,
    notebook05_output_dir
    / "partA_feature_scaler.joblib"
)

joblib.dump(
    reliability_svd,
    notebook05_output_dir
    / "reliability_morgan_svd.joblib"
)

joblib.dump(
    reliability_scaler,
    notebook05_output_dir
    / "reliability_feature_scaler.joblib"
)


print(
    "Fitted SVD and scaling "
    "transformations exported."
)

Fitted SVD and scaling transformations exported.


In [84]:
# ============================================================
# Export five-member reliability ensemble
# ============================================================

model_dir = (
    notebook05_output_dir
    / "reliability_models"
)

model_dir.mkdir(
    exist_ok=True
)


for seed, model in (
    reliability_models.items()
):

    torch.save(
        model.state_dict(),
        model_dir
        / (
            f"multitask_reliability_"
            f"seed_{seed}.pt"
        )
    )


print(
    f"Reliability models exported: "
    f"{len(reliability_models)}"
)

Reliability models exported: 5


In [85]:
# ============================================================
# Export final conformal thresholds
# ============================================================

conformal_threshold_table = pd.DataFrame({

    "Target":
        core_targets,

    "Absolute_Conformal_HalfWidth":
        [
            absolute_conformal_thresholds[
                target
            ]
            for target in core_targets
        ],

    "Scaled_Conformal_q":
        [
            scaled_conformal_thresholds[
                target
            ]
            for target in core_targets
        ],

    "Nominal_Coverage":
        TARGET_COVERAGE
})


conformal_threshold_table.to_csv(
    notebook05_output_dir
    / "conformal_thresholds.csv",
    index=False
)


print(
    "Conformal thresholds exported."
)

display(
    conformal_threshold_table.round(4)
)

Conformal thresholds exported.


,Target,Absolute_Conformal_HalfWidth,Scaled_Conformal_q,Nominal_Coverage
0,Tc,0.0678,15.3779,0.9
1,Density,0.0637,5.7097,0.9
2,Rg,6.2292,14.5211,0.9


In [86]:
# ============================================================
# Final Notebook 05 configuration
# ============================================================

notebook05_config = {

    "project":
        "Polymer Informatics Atlas",

    "notebook":
        "05_shared_learning_and_reliability",

    "core_targets":
        core_targets,

    "random_state_partA":
        RANDOM_STATE,

    "random_state_reliability":
        RELIABILITY_RANDOM_STATE,

    "partA_split": {
        "train_fraction":
            0.70,

        "validation_fraction":
            0.15,

        "test_fraction":
            0.15
    },

    "reliability_split": {
        "train_fraction":
            0.65,

        "validation_fraction":
            0.15,

        "calibration_fraction":
            0.10,

        "test_fraction":
            0.10
    },

    "input_representation": {

        "interpretable_features":
            36,

        "morgan_source":
            "count Morgan radius-4 / 2048",

        "morgan_svd_components":
            N_MORGAN_COMPONENTS,

        "final_dimensions":
            int(
                X_neural.shape[1]
            )
    },

    "multitask_architecture": {

        "shared_hidden_layers":
            [
                256,
                128,
                64
            ],

        "task_heads":
            len(core_targets),

        "activation":
            "ReLU",

        "dropout":
            [
                0.20,
                0.15
            ]
    },

    "optimizer": {

        "name":
            "AdamW",

        "learning_rate":
            LEARNING_RATE,

        "weight_decay":
            WEIGHT_DECAY
    },

    "training": {

        "batch_size":
            BATCH_SIZE,

        "maximum_epochs":
            MAX_EPOCHS,

        "early_stopping_patience":
            PATIENCE,

        "random_seeds":
            NEURAL_SEEDS,

        "masked_task_balanced_loss":
            True
    },

    "shared_learning": {

        "complete_triplets":
            int(
                len(
                    core_complete_idx
                )
            ),

        "polymers_with_at_least_one_core_target":
            int(
                len(
                    core_union_idx
                )
            ),

        "masked_learning":
            True,

        "complete_case_ablation":
            True
    },

    "conformal_prediction": {

        "nominal_coverage":
            TARGET_COVERAGE,

        "alpha":
            CONFORMAL_ALPHA,

        "primary_method":
            "absolute-residual split conformal",

        "adaptive_method":
            (
                "ensemble-disagreement-scaled "
                "split conformal"
            ),

        "primary_method_selected":
            "absolute-residual split conformal",

        "reason":
            (
                "Adaptive disagreement-scaled intervals "
                "were wider on average and did not improve "
                "the global coverage-efficiency trade-off."
            )
    },

    "final_reliability_outputs": [

        "ensemble mean prediction",

        "90% fixed split-conformal interval",

        "ensemble prediction standard deviation",

        (
            "maximum training-set binary Morgan "
            "Tanimoto similarity"
        )
    ],

    "principal_transfer_conclusion":
        (
            "Masked multitask learning clearly "
            "outperformed complete-case multitask "
            "learning, demonstrating value from "
            "partially labelled polymers. Gains over "
            "matched single-task neural models were "
            "small and seed dependent."
        ),

    "principal_reliability_conclusion":
        (
            "Predictive reliability is multidimensional "
            "and property dependent. Ensemble disagreement "
            "was operationally useful for Tc and Density, "
            "while chemical support was comparatively more "
            "informative for Rg. Fixed split-conformal "
            "prediction provided the calibrated uncertainty "
            "layer."
        )
}


with open(
    notebook05_output_dir
    / "notebook05_config.json",
    "w"
) as f:

    json.dump(
        notebook05_config,
        f,
        indent=2
    )


print(
    "Notebook 05 final configuration exported."
)

Notebook 05 final configuration exported.


In [87]:
# ============================================================
# Create final Notebook 05 archive
# ============================================================

archive_path = shutil.make_archive(
    "polymer_atlas_notebook05_outputs",
    "zip",
    root_dir=notebook05_output_dir
)


print(
    "Notebook 05 archive created:"
)

print(
    archive_path
)

Notebook 05 archive created:
/content/polymer_atlas_notebook05_outputs.zip


In [88]:
# ============================================================
# Final Notebook 05 export audit
# ============================================================

print(
    "NOTEBOOK 05 FINAL EXPORT AUDIT"
)

print(
    "=" * 100
)


exported_files = sorted(
    notebook05_output_dir.rglob(
        "*"
    )
)

file_count = 0
total_size = 0


for path in exported_files:

    if path.is_file():

        file_count += 1

        file_size = (
            path.stat().st_size
        )

        total_size += (
            file_size
        )

        relative_path = (
            path.relative_to(
                notebook05_output_dir
            )
        )

        print(
            f"{str(relative_path):62s} "
            f"{file_size / 1024:.2f} KB"
        )


print(
    "\n"
    + "=" * 100
)

print(
    f"Total exported files: "
    f"{file_count}"
)

print(
    f"Total output size: "
    f"{total_size / (1024**2):.2f} MB"
)

print(
    "\nNotebook 05 scientific "
    "workflow complete."
)

NOTEBOOK 05 FINAL EXPORT AUDIT
Density_adaptive_conformal_test.csv                            11.00 KB
Density_calibration_predictions.csv                            6.13 KB
Density_chemistry_reliability_test.csv                         8.29 KB
Density_fixed_conformal_test.csv                               10.99 KB
Density_reliability_test_predictions.csv                       6.96 KB
Rg_adaptive_conformal_test.csv                                 10.68 KB
Rg_calibration_predictions.csv                                 5.73 KB
Rg_chemistry_reliability_test.csv                              8.05 KB
Rg_fixed_conformal_test.csv                                    10.69 KB
Rg_reliability_test_predictions.csv                            6.70 KB
Tc_adaptive_conformal_test.csv                                 11.82 KB
Tc_calibration_predictions.csv                                 8.19 KB
Tc_chemistry_reliability_test.csv                              8.81 KB
Tc_fixed_conformal_test.csv              

In [89]:
from google.colab import files

files.download(
    "polymer_atlas_notebook05_outputs.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>